In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
DATA_DIR = Path("./checkpoints/alexnet_long_stream/")

# Configure experiments: specify the list of seeds and filename templates
EXPERIMENTS = [
    {
        "label": r"Heavy-Tailed ($\\\\alpha=1.2$)",
        "seeds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],  # Add all your seed integers here
        "tasks_pattern": "gpm_alexnet_a1.2_s{seed}_tasks.csv",
        "curves_pattern": "gpm_alexnet_a1.2_s{seed}_curves.csv",
        "fallback_single_pattern": "gpm_alexnet_a1.2_s{seed}.csv",
        "color": "#1f77b4",
    },
    {
        "label": r"Gaussian Baseline ($\\\\alpha=2.0$)",
        "seeds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        "tasks_pattern": "gpm_alexnet_a2.0_s{seed}_tasks.csv",
        "curves_pattern": "gpm_alexnet_a2.0_s{seed}_curves.csv",
        "fallback_single_pattern": "gpm_alexnet_a2.0_s{seed}.csv",
        "color": "#ff7f0e",
    },
]


def load_single_seed_data(exp_cfg, seed):
    """Loads tasks and curves DataFrames for a single seed."""
    tasks_file = exp_cfg["tasks_pattern"].format(seed=seed)
    curves_file = exp_cfg["curves_pattern"].format(seed=seed)
    single_file = exp_cfg["fallback_single_pattern"].format(seed=seed)

    tasks_path = DATA_DIR / tasks_file
    curves_path = DATA_DIR / curves_file
    single_path = DATA_DIR / single_file

    df_tasks = None
    if tasks_path.exists():
        df_tasks = pd.read_csv(tasks_path)
    elif single_path.exists():
        df_single = pd.read_csv(single_path)
        df_tasks = df_single.dropna(subset=["total_basis_rank"]).reset_index(
            drop=True
        )

    df_curves = None
    if curves_path.exists():
        df_curves = pd.read_csv(curves_path)
    elif single_path.exists():
        df_single = pd.read_csv(single_path)
        if "intra_task_acc" in df_single.columns:
            df_curves = df_single.dropna(subset=["intra_task_acc"]).reset_index(
                drop=True
            )

    return df_tasks, df_curves


def compute_seen_mean_acc(df_tasks):
    num_tasks = len(df_tasks)
    mean_accs = []
    for t_idx in range(num_tasks):
        acc_cols = [f"task_{j}_acc" for j in range(t_idx + 1)]
        row_mean = df_tasks.loc[t_idx, acc_cols].mean() * 100.0
        mean_accs.append(row_mean)
    return np.array(mean_accs)


def compute_task_auc(df_curves, num_tasks=20):
    if df_curves is None or "intra_task_acc" not in df_curves.columns:
        return None
    auc_per_task = []
    # Compatible with both NumPy 2.0+ (np.trapezoid) and older versions (np.trapz)
    trapz_fn = getattr(np, "trapezoid", np.trapezoid)

    for t_idx in range(num_tasks):
        task_data = df_curves[df_curves["task_idx"] == t_idx].sort_values(
            by="epoch"
        )
        if len(task_data) > 0:
            epochs = task_data["epoch"].values
            accs = task_data["intra_task_acc"].values
            auc = trapz_fn(accs, epochs) / (epochs[-1] - epochs[0])
            auc_per_task.append(auc * 100.0)
        else:
            auc_per_task.append(np.nan)
    return np.array(auc_per_task)


def compute_bwt_fwt_trajectories(df_tasks, num_tasks=20):
    """Computes Backward Transfer (BWT) and Zero-Shot Forward Transfer (FWT) across stream."""
    acc_matrix = np.zeros((num_tasks, num_tasks))
    for i in range(num_tasks):
        for j in range(num_tasks):
            col = f"task_{j}_acc"
            if col in df_tasks.columns and pd.notna(df_tasks.loc[i, col]):
                acc_matrix[i, j] = df_tasks.loc[i, col]

    # 1. Backward Transfer (BWT) after each task t (for t >= 1)
    bwt_traj = [np.nan]  # No BWT after Task 1
    for t in range(1, num_tasks):
        forgetting = [acc_matrix[t, j] - acc_matrix[j, j] for j in range(t)]
        bwt_traj.append(np.mean(forgetting) * 100.0)

    # 2. Zero-Shot Forward Transfer (FWT) after each task t (for t < num_tasks - 1)
    fwt_traj = []
    for t in range(num_tasks - 1):
        zero_shot_future = [acc_matrix[t, j] for j in range(t + 1, num_tasks)]
        fwt_traj.append(np.mean(zero_shot_future) * 100.0)
    fwt_traj.append(np.nan)

    return np.array(bwt_traj), np.array(fwt_traj)


# --- PLOTTING 2x2 GRID ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
axes = axes.flatten()

for exp in EXPERIMENTS:
    seen_acc_runs = []
    auc_runs = []
    rank_runs = []
    bwt_runs = []
    fwt_runs = []
    num_tasks = None

    for seed in exp["seeds"]:
        df_tasks, df_curves = load_single_seed_data(exp, seed)
        if df_tasks is None:
            print(f"Warning: Skipping seed {seed} for {exp['label']} (Tasks file not found).")
            continue

        if num_tasks is None:
            num_tasks = len(df_tasks)

        # 1. Seen Acc
        seen_acc_runs.append(compute_seen_mean_acc(df_tasks))

        # 2. AUC
        auc_val = compute_task_auc(df_curves, num_tasks=num_tasks)
        if auc_val is not None:
            auc_runs.append(auc_val)

        # 3. Cumulative Basis Rank
        rank_runs.append(df_tasks["total_basis_rank"].values)

        # 4. Transfer Dynamics
        bwt, fwt = compute_bwt_fwt_trajectories(df_tasks, num_tasks=num_tasks)
        bwt_runs.append(bwt)
        fwt_runs.append(fwt)

    if not seen_acc_runs:
        print(f"Error: No valid seed data found for {exp['label']}")
        continue

    task_indices = np.arange(1, num_tasks + 1)
    color = exp["color"]
    label = exp["label"]

    # Helper: calculate mean and std safely ignoring NaNs
    def get_stats(data_list):
        arr = np.array(data_list)
        return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)

    # 1. Average Retained Accuracy
    mean_acc, std_acc = get_stats(seen_acc_runs)
    axes[0].plot(task_indices, mean_acc, marker="o", label=label, color=color, linewidth=2)
    axes[0].fill_between(task_indices, mean_acc - std_acc, mean_acc + std_acc, color=color, alpha=0.18)

    # 2. Intra-Task Learning AUC
    if auc_runs:
        mean_auc, std_auc = get_stats(auc_runs)
        axes[1].plot(task_indices, mean_auc, marker="s", label=label, color=color, linewidth=2)
        axes[1].fill_between(task_indices, mean_auc - std_auc, mean_auc + std_auc, color=color, alpha=0.18)

    # 3. Cumulative GPM Basis Rank
    mean_rank, std_rank = get_stats(rank_runs)
    axes[2].plot(task_indices, mean_rank, marker="^", label=label, color=color, linewidth=2)
    axes[2].fill_between(task_indices, mean_rank - std_rank, mean_rank + std_rank, color=color, alpha=0.18)

    # 4. Backward Transfer & Forward Transfer
    mean_bwt, std_bwt = get_stats(bwt_runs)
    mean_fwt, std_fwt = get_stats(fwt_runs)

    # BWT (Solid)
    axes[3].plot(task_indices, mean_bwt, marker="v", linestyle="-", label=f"{label} (BWT)", color=color, linewidth=2)
    axes[3].fill_between(task_indices, mean_bwt - std_bwt, mean_bwt + std_bwt, color=color, alpha=0.15)

    # FWT (Dashed)
    axes[3].plot(task_indices, mean_fwt, marker="x", linestyle="--", label=f"{label} (Zero-Shot FWT)", color=color, alpha=0.75, linewidth=1.5)
    axes[3].fill_between(task_indices, mean_fwt - std_fwt, mean_fwt + std_fwt, color=color, alpha=0.08)

# --- FORMATTING PLOTS ---
axes[0].set_title("Average Retained Accuracy (Seen Tasks)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Completed Task Index")
axes[0].set_ylabel("Mean Accuracy (%)")
axes[0].set_xticks(range(1, 51, 5))
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend(loc="lower left", frameon=True)

axes[1].set_title("Intra-Task Learning AUC (Convergence Speed)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Task Index")
axes[1].set_ylabel("Normalized AUC (%)")
axes[1].set_xticks(range(1, 51, 5))
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend(loc="lower right", frameon=True)

axes[2].set_title("Cumulative GPM Basis Rank (Subspace Growth)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Completed Task Index")
axes[2].set_ylabel("Total Stored Rank")
axes[2].set_xticks(range(1, 51, 5))
axes[2].grid(True, linestyle="--", alpha=0.6)
axes[2].legend(loc="upper left", frameon=True)

axes[3].set_title("Transfer Dynamics (BWT & Zero-Shot FWT)", fontsize=12, fontweight="bold")
axes[3].set_xlabel("Completed Task Index")
axes[3].set_ylabel("Transfer Metric (%)")
axes[3].set_xticks(range(1, 51, 5))
axes[3].axhline(0, color="gray", linestyle=":", alpha=0.7)
axes[3].grid(True, linestyle="--", alpha=0.6)
axes[3].legend(loc="center left", fontsize=9, frameon=True)

plt.tight_layout()
output_plot_path = DATA_DIR / "gpm_comprehensive_transfer_dynamics.png"
plt.savefig(output_plot_path, dpi=300)
plt.show()
print(f"2x2 Multi-seed diagnostic plot successfully saved to: {output_plot_path}")

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# --- CONFIGURATION & DATA PATHS ---
DATA_DIR = Path("./checkpoints/alexnet_long_stream/")

EXPERIMENTS = [
    {
        "name": "heavy_tailed",
        "label": r"Heavy-Tailed ($\alpha=1.2$)",
        "seeds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        "tasks_pattern": "gpm_alexnet_a1.2_s{seed}_tasks.csv",
        "curves_pattern": "gpm_alexnet_a1.2_s{seed}_curves.csv",
        "fallback_single_pattern": "gpm_alexnet_a1.2_s{seed}.csv",
        "color": "#1f77b4",
    },
    {
        "name": "gaussian",
        "label": r"Gaussian Baseline ($\alpha=2.0$)",
        "seeds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        "tasks_pattern": "gpm_alexnet_a2.0_s{seed}_tasks.csv",
        "curves_pattern": "gpm_alexnet_a2.0_s{seed}_curves.csv",
        "fallback_single_pattern": "gpm_alexnet_a2.0_s{seed}.csv",
        "color": "#ff7f0e",
    },
]


def load_bwt_series(exp_cfg, data_dir=DATA_DIR):
    """Loads BWT trajectories across seeds for a given experiment configuration."""
    all_bwt = []
    valid_seeds = []

    for seed in exp_cfg["seeds"]:
        tasks_file = data_dir / exp_cfg["tasks_pattern"].format(seed=seed)
        fallback_file = data_dir / exp_cfg["fallback_single_pattern"].format(
            seed=seed
        )

        target_file = (
            tasks_file
            if tasks_file.exists()
            else (fallback_file if fallback_file.exists() else None)
        )
        if target_file is None:
            print(f"Warning: File missing for seed {seed} ({exp_cfg['label']})")
            continue

        df = pd.read_csv(target_file)

        # Detect BWT column or compute from accuracy matrix if present
        bwt_col = [c for c in df.columns if "bwt" in c.lower()]
        if bwt_col:
            bwt_series = df[bwt_col[0]].to_numpy()
        else:
            # If full R_{i,j} table exists, compute BWT manually:
            # BWT_t = 1/(t-1) * sum_{i=1}^{t-1} (R_{t,i} - R_{i,i})
            acc_cols = [
                c
                for c in df.columns
                if c.startswith("acc_task_") or c.startswith("task_")
            ]
            if len(acc_cols) > 1:
                R = df[acc_cols].to_numpy()
                bwt_series = np.full(len(R), np.nan)
                for t in range(1, len(R)):
                    diag_accs = np.diag(R[:t, :t])
                    curr_accs = R[t, :t]
                    bwt_series[t] = np.mean(curr_accs - diag_accs)
            else:
                raise KeyError(
                    f"Could not locate BWT or task accuracy columns in {target_file.name}"
                )

        all_bwt.append(bwt_series)
        valid_seeds.append(seed)

    return np.array(all_bwt), valid_seeds


# --- 1. LOAD DATA ---
results = {}
for exp in EXPERIMENTS:
    bwt_mat, valid_seeds = load_bwt_series(exp)
    results[exp["name"]] = {
        "bwt": bwt_mat,  # Shape: [num_seeds, num_tasks]
        "seeds": valid_seeds,
        "label": exp["label"],
        "color": exp["color"],
    }

# --- 2. DEDICATED HIGH-RESOLUTION BWT PLOT ---
fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)

# BWT is formally defined starting from Task 2 (after completing second task)
task_indices = np.arange(1, results["heavy_tailed"]["bwt"].shape[1] + 1)
start_idx = 1  # Start at Task 2 index

for key, data in results.items():
    bwt_arr = data["bwt"][:, start_idx:]
    t_axis = task_indices[start_idx:]

    mean_bwt = np.nanmean(bwt_arr, axis=0) * 100.0  # Convert to percentage
    sem_bwt = np.nanstd(bwt_arr, axis=0, ddof=1) / np.sqrt(bwt_arr.shape[0]) * 100.0  # Standard Error of the Mean

    # Plot mean trajectory
    ax.plot(
        t_axis,
        mean_bwt,
        label=data["label"],
        color=data["color"],
        lw=2.2,
        marker="o",
        markersize=5,
    )
    # 95% Confidence Interval (1.96 * SEM)
    ax.fill_between(
        t_axis,
        mean_bwt - 1.96 * sem_bwt,
        mean_bwt + 1.96 * sem_bwt,
        color=data["color"],
        alpha=0.18,
    )

# Zero reference line (Neutral Transfer boundary)
ax.axhline(
    0.0,
    color="black",
    linestyle="--",
    lw=1.2,
    alpha=0.7,
    label="Zero Forgetting (BWT = 0.0)",
)

ax.set_title(
    "Backward Transfer (BWT) Trajectory Across Continual Tasks",
    fontsize=12,
    fontweight="bold",
)
ax.set_xlabel("Completed Task Index $T$", fontsize=11)
ax.set_ylabel("Backward Transfer (%)", fontsize=11)
ax.set_xticks(task_indices[start_idx:])
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=10, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()

# --- 3. STATISTICAL SIGNIFICANCE TESTS ---
print("=" * 65)
print("STATISTICAL SIGNIFICANCE TESTS FOR BACKWARD TRANSFER (BWT)")
print("=" * 65)

ht_final = results["heavy_tailed"]["bwt"][:, -1]
gauss_final = results["gaussian"]["bwt"][:, -1]

# A. One-Sample Test: Is HT BWT strictly greater than zero at Task 20?
t_stat_ht0, p_val_ht0 = stats.ttest_1samp(ht_final, popmean=0.0)
w_stat_ht0, p_val_w_ht0 = stats.wilcoxon(
    ht_final, alternative="greater"
)  # non-parametric

print(f"\n1. Heavy-Tailed vs. Zero Baseline (Task 20):")
print(
    f"   Mean BWT: {np.mean(ht_final):.3f}% ± {np.std(ht_final, ddof=1):.3f}% (N={len(ht_final)})"
)
print(f"   t-test against 0.0: t = {t_stat_ht0:.3f}, p-value = {p_val_ht0:.4e}")
print(
    f"   Wilcoxon Signed-Rank (one-tailed > 0): W = {w_stat_ht0}, p-value = {p_val_w_ht0:.4e}"
)

# B. Paired Test: Does HT produce significantly higher BWT than Gaussian?
diff = ht_final - gauss_final
t_stat_pair, p_val_pair = stats.ttest_rel(ht_final, gauss_final)
w_stat_pair, p_val_w_pair = stats.wilcoxon(
    ht_final, gauss_final, alternative="greater"
)

print(f"\n2. Heavy-Tailed vs. Gaussian Baseline (Task 20):")
print(f"   Gaussian Mean BWT: {np.mean(gauss_final):.3f}%")
print(f"   Mean Improvement (Δ): +{np.mean(diff):.3f}%")
print(
    f"   Paired t-test: t = {t_stat_pair:.3f}, p-value = {p_val_pair:.4e} (two-tailed)"
)
print(
    f"   Wilcoxon Paired Test: W = {w_stat_pair}, p-value = {p_val_w_pair:.4e} (one-tailed)"
)

# C. Trend Analysis: Linear slope test across Task 2 -> 20 per seed
slopes_ht = [
    stats.linregress(
        np.arange(len(s[start_idx:])), s[start_idx:]
    ).slope
    for s in results["heavy_tailed"]["bwt"]
]
slopes_gauss = [
    stats.linregress(
        np.arange(len(s[start_idx:])), s[start_idx:]
    ).slope
    for s in results["gaussian"]["bwt"]
]

t_stat_slope, p_val_slope = stats.ttest_1samp(slopes_ht, popmean=0.0)
print(f"\n3. Upward Drift Slope Test (Tasks 2 -> 20):")
print(f"   HT Mean Slope: {np.mean(slopes_ht):+.4f}% per task (t = {t_stat_slope:.3f}, p = {p_val_slope:.4e})")
print(f"   Gaussian Mean Slope: {np.mean(slopes_gauss):+.4f}% per task")
print("=" * 65)

In [ ]:
def plot_scree_comparison(
    gaussian_snapshot_path,
    ht_snapshot_path,
    layer_key,
    save_path=None,
    normalize=True,
    cumulative=True,
):
    """Loads a Gaussian and a Heavy-Tailed snapshot, extracts singular values

    for a specified layer, and plots comparative scree and cumulative energy curves.
    """
    # 1. Load snapshot checkpoints
    snap_gauss = torch.load(gaussian_snapshot_path, map_location="cpu")
    snap_ht = torch.load(ht_snapshot_path, map_location="cpu")

    # Clean layer key format to match snapshot dict convention
    clean_layer_key = str(layer_key).replace(".", "_")

    if clean_layer_key not in snap_gauss["svd"]:
        available = list(snap_gauss["svd"].keys())
        raise KeyError(
            f"Layer '{clean_layer_key}' not found in snapshot SVD data. Available: {available}"
        )

    # 2. Extract singular values
    s_gauss = snap_gauss["svd"][clean_layer_key]["S"].numpy()
    s_ht = snap_ht["svd"][clean_layer_key]["S"].numpy()

    # Metadata for labels
    meta_gauss = snap_gauss.get("metadata", {})
    meta_ht = snap_ht.get("metadata", {})

    alpha_gauss = meta_gauss.get("alpha", 2.0)
    alpha_ht = meta_ht.get("alpha", "< 2.0")
    task_num = meta_gauss.get("task_number", meta_gauss.get("task_idx", "?"))

    # Optional normalization by top singular value (sigma_i / sigma_1)
    if normalize:
        s_gauss_plot = s_gauss / s_gauss[0]
        s_ht_plot = s_ht / s_ht[0]
        y_label = r"Normalized Singular Value $\sigma_i / \sigma_1$"
    else:
        s_gauss_plot = s_gauss
        s_ht_plot = s_ht
        y_label = r"Singular Value $\sigma_i$"

    # 3. Plotting
    num_panels = 2 if cumulative else 1
    fig, axes = plt.subplots(1, num_panels, figsize=(6.5 * num_panels, 4.5), dpi=300)
    if not cumulative:
        axes = [axes]

    # Panel 1: Scree Plot (Log-Linear)
    ax1 = axes[0]
    rank_gauss = np.arange(1, len(s_gauss_plot) + 1)
    rank_ht = np.arange(1, len(s_ht_plot) + 1)

    ax1.plot(
        rank_gauss,
        s_gauss_plot,
        label=rf"Gaussian ($\\\\alpha={alpha_gauss}$)",
        color="#1f77b4",
        lw=2,
    )
    ax1.plot(
        rank_ht,
        s_ht_plot,
        label=rf"Heavy-Tailed ($\\\\alpha={alpha_ht}$)",
        color="#d62728",
        lw=2,
    )

    ax1.set_yscale("log")
    ax1.set_xlabel("Singular Value Index $i$", fontsize=11)
    ax1.set_ylabel(y_label, fontsize=11)
    ax1.set_title(f"Layer: {layer_key} (Task {task_num}) - Spectral Decay", fontsize=12)
    ax1.grid(True, which="both", ls="--", alpha=0.4)
    ax1.legend(frameon=True, fontsize=10)

    # Panel 2: Cumulative Energy Explained
    if cumulative:
        ax2 = axes[1]
        cum_energy_gauss = np.cumsum(s_gauss**2) / np.sum(s_gauss**2)
        cum_energy_ht = np.cumsum(s_ht**2) / np.sum(s_ht**2)

        ax2.plot(
            rank_gauss,
            cum_energy_gauss,
            label=rf"Gaussian ($\\\\alpha={alpha_gauss}$)",
            color="#1f77b4",
            lw=2,
        )
        ax2.plot(
            rank_ht,
            cum_energy_ht,
            label=rf"Heavy-Tailed ($\\\\alpha={alpha_ht}$)",
            color="#d62728",
            lw=2,
        )

        # Standard GPM energy thresholds
        for th in [0.95, 0.99]:
            ax2.axhline(
                th,
                color="black",
                ls=":",
                alpha=0.6,
                label=f"Threshold {int(th * 100)}%" if th == 0.95 else None,
            )

        ax2.set_xlabel("Basis Rank $k$", fontsize=11)
        ax2.set_ylabel(
            r"Cumulative Energy $\sum_{i=1}^k \sigma_i^2 / \|\mathbf{R}\|_F^2$",
            fontsize=11,
        )
        ax2.set_title(
            f"Layer: {layer_key} (Task {task_num}) - Basis Capture", fontsize=12
        )
        ax2.set_ylim(None, 1.02)
        ax2.grid(True, ls="--", alpha=0.4)
        ax2.legend(frameon=True, fontsize=10)

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved scree plot to {save_path}")

    plt.show()


# Example usage:
plot_scree_comparison(
    gaussian_snapshot_path="checkpoints/snapshots/snapshot_A2.0_T20_s0.pt",
    ht_snapshot_path="checkpoints/snapshots/snapshot_A1.2_T20_s0.pt",
    layer_key="4",  # or "layer1", "fc1", etc.
    # save_path="plots/scree_layer0_task5.pdf"
)

In [ ]:
import re
from pathlib import Path
from typing import Any, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import ticker
from scipy.signal import savgol_filter


def resolve_layer_keys_by_index(svd_dict, basis_dict, layer_query):
    """Maps SVD and Basis keys strictly by layer ordinal index."""
    svd_keys = list(svd_dict.keys())
    basis_keys = list(basis_dict.keys())

    # Map by direct index if querying by integer or matching string
    if isinstance(layer_query, int):
        idx = layer_query
    elif "classifier" in str(layer_query).lower() or "head" in str(layer_query).lower():
        idx = -1
    else:
        # Extract digits: if 'features_8_weight', layer index in 10-layer stack is 8//2 = 4
        digits = re.findall(r"\d+", str(layer_query))
        num = int(digits[0]) if digits else 0
        idx = (
            num // 2
            if "features" in str(layer_query) and num % 2 == 0
            else min(num, len(svd_keys) - 1)
        )

    return svd_keys[idx], basis_keys[idx]


def compute_mode_alignment_from_svd(
    U_t: Union[torch.Tensor, np.ndarray],
    S_t: Union[torch.Tensor, np.ndarray],
    M_prev: Optional[Union[torch.Tensor, np.ndarray]],
    max_modes: int = 50,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Computes mode projection overlap p_i and energy-weighted alignment E_i."""
    if isinstance(U_t, torch.Tensor):
        U_t = U_t.detach().cpu().numpy()
    if isinstance(S_t, torch.Tensor):
        S_t = S_t.detach().cpu().numpy()
    if isinstance(M_prev, torch.Tensor):
        M_prev = M_prev.detach().cpu().numpy()

    U_t = np.asarray(U_t, dtype=np.float64)
    S_t = np.asarray(S_t, dtype=np.float64)

    num_modes = min(len(S_t), U_t.shape[1], max_modes)
    U_sub = U_t[:, :num_modes]
    sigmas_sq = S_t[:num_modes] ** 2

    if M_prev is None or M_prev.size == 0 or (M_prev.ndim > 1 and M_prev.shape[1] == 0):
        p_i = np.zeros(num_modes, dtype=np.float64)
    else:
        M_prev = np.asarray(M_prev, dtype=np.float64)
        if M_prev.ndim == 1:
            M_prev = M_prev[:, np.newaxis]

        if M_prev.shape[0] != U_sub.shape[0]:
            raise ValueError(
                f"Dimension mismatch: M_prev has ambient dim {M_prev.shape[0]}, "
                f"but U_t has ambient dim {U_sub.shape[0]}."
            )

        proj_coords = M_prev.T @ U_sub  # Shape: (K_prev, num_modes)
        p_i = np.sum(proj_coords**2, axis=0)
        p_i = np.clip(p_i, 0.0, 1.0)

    E_i = sigmas_sq * p_i
    return p_i, sigmas_sq, E_i


def load_snapshots_for_regime(
    snapshot_dir: Union[str, Path],
    alpha: float,
    seed: Optional[int] = 0,
    total_tasks: int = 20,
    epoch: Optional[int] = None,
) -> dict[int, dict[str, Any]]:
    """Loads snapshot checkpoint files mapped by task index."""
    snapshot_dir = Path(snapshot_dir)
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory does not exist: {snapshot_dir}")

    snapshots_by_task = {}
    epoch_tag = f"_E{epoch}" if epoch is not None else ""
    seed_tag = f"_s{seed}" if seed is not None else "*_s*"

    for t_num in range(total_tasks + 2):
        pattern = f"snapshot_A{alpha}_T{t_num:02d}{epoch_tag}{seed_tag}.pt"
        matches = sorted(snapshot_dir.glob(pattern))

        if not matches:
            fallback = f"*A{alpha}*T{t_num:02d}*{epoch_tag}{seed_tag}*.pt"
            matches = sorted(snapshot_dir.glob(fallback))

        for file_path in matches:
            data = torch.load(file_path, map_location="cpu", weights_only=False)
            t_idx = data["metadata"]["task_idx"]
            snapshots_by_task[t_idx] = data

    return snapshots_by_task


def run_temporal_mode_alignment_analysis(
    snapshot_dir: Union[str, Path],
    alpha: float,
    seed: Optional[int] = 0,
    layer_name: Union[str, int] = "features_8_weight",
    total_tasks: int = 20,
    max_modes: int = 40,
) -> dict[str, Any]:
    """Extracts mode alignment metrics across sequential tasks."""
    snapshots = load_snapshots_for_regime(
        snapshot_dir=snapshot_dir,
        alpha=alpha,
        seed=seed,
        total_tasks=total_tasks,
    )

    available_tasks = sorted(snapshots.keys())
    eval_tasks = [t for t in available_tasks if t >= 1 and (t - 1) in snapshots]

    if not eval_tasks:
        raise ValueError(
            f"No consecutive task pairs found for alpha={alpha}, seed={seed} in {snapshot_dir}"
        )

    tasks_out, p_i_list, sigmas_sq_list, E_i_list = [], [], [], []
    last_svd_k, last_basis_k = None, None

    for t in eval_tasks:
        snap_current = snapshots[t]
        snap_prev = snapshots[t - 1]

        svd_k, basis_k = resolve_layer_keys_by_index(
            snap_current["svd"], snap_prev["current_basis"], layer_name
        )
        last_svd_k, last_basis_k = svd_k, basis_k

        U_t = snap_current["svd"][svd_k]["U"]
        S_t = snap_current["svd"][svd_k]["S"]
        M_prev = snap_prev["current_basis"][basis_k]

        p_i, sigmas_sq, E_i = compute_mode_alignment_from_svd(
            U_t=U_t, S_t=S_t, M_prev=M_prev, max_modes=max_modes
        )

        tasks_out.append(t)
        p_i_list.append(p_i)
        sigmas_sq_list.append(sigmas_sq)
        E_i_list.append(E_i)

    return {
        "alpha": alpha,
        "tasks": np.array(tasks_out, dtype=int),
        "p_i": np.array(p_i_list),
        "sigmas_sq": np.array(sigmas_sq_list),
        "E_i": np.array(E_i_list),
        "resolved_svd_key": last_svd_k,
        "resolved_basis_key": last_basis_k,
    }


def smooth_series(
    y: np.ndarray, window_length: int = 15, polyorder: int = 2
) -> np.ndarray:
    """Applies Savitzky-Golay smoothing with dynamic window sizing."""
    n = len(y)
    if n < 5:
        return y
    w = min(window_length, n if n % 2 != 0 else n - 1)
    if w <= polyorder:
        w = polyorder + 2 if (polyorder + 2) % 2 != 0 else polyorder + 3
    if w > n:
        return y
    return savgol_filter(y, window_length=w, polyorder=polyorder)


def plot_mode_alignment_publication(
    res_ht: dict[str, Any],
    res_gauss: dict[str, Any],
    task_idx: int = 1,
    max_modes: int = 150,
    smooth_window: int = 15,
    save_path: Optional[Union[str, Path]] = None,
) -> None:
    """Generates a 3-panel publication-grade breakdown:

    1. Geometric Overlap (p_i) with smoothing.
    2. Cumulative Energy Fraction (reveals how fast each regime accumulates total power).
    3. Mode-Wise Projected Power Contribution (w_i * p_i).
    """
    ht_matches = np.where(res_ht["tasks"] == task_idx)[0]
    gauss_matches = np.where(res_gauss["tasks"] == task_idx)[0]

    if len(ht_matches) == 0 or len(gauss_matches) == 0:
        raise ValueError(f"Task {task_idx} not found in results.")

    ht_t = ht_matches[0]
    g_t = gauss_matches[0]

    k = min(max_modes, res_ht["p_i"].shape[1], res_gauss["p_i"].shape[1])
    modes = np.arange(1, k + 1)

    # 1. Raw & Smoothed Subspace Overlaps
    ht_p_raw = res_ht["p_i"][ht_t, :k]
    g_p_raw = res_gauss["p_i"][g_t, :k]

    ht_p_smooth = smooth_series(ht_p_raw, window_length=smooth_window)
    g_p_smooth = smooth_series(g_p_raw, window_length=smooth_window)

    # 2. Normalized Variance & Cumulative Energy
    ht_sig_sq = res_ht["sigmas_sq"][ht_t, :k]
    g_sig_sq = res_gauss["sigmas_sq"][g_t, :k]

    ht_var_frac = ht_sig_sq / res_ht["sigmas_sq"][ht_t].sum()
    g_var_frac = g_sig_sq / res_gauss["sigmas_sq"][g_t].sum()

    ht_cum_energy = np.cumsum(ht_var_frac)
    g_cum_energy = np.cumsum(g_var_frac)

    # 3. Mode-Wise Projected Power Contribution
    ht_proj_power = ht_var_frac * ht_p_raw
    g_proj_power = g_var_frac * g_p_raw

    # Setup Canvas: 3 Panels
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    c_ht = "#1f77b4"
    c_gauss = "#d62728"

    # =========================================================================
    # Panel 1: Geometric Subspace Overlap (p_i)
    # =========================================================================
    # Raw faint traces
    ax1.plot(modes, ht_p_raw, color=c_ht, alpha=0.22, lw=1.0)
    ax1.plot(modes, g_p_raw, color=c_gauss, alpha=0.22, lw=1.0)

    # Bold smoothed trends
    ax1.plot(
        modes,
        ht_p_smooth,
        color=c_ht,
        lw=2.5,
        label=rf"Heavy-Tailed ($\\\\alpha={res_ht.get('alpha', 1.2)}$)",
    )
    ax1.plot(
        modes,
        g_p_smooth,
        color=c_gauss,
        lw=2.5,
        label=rf"Gaussian ($\\\\alpha={res_gauss.get('alpha', 2.0)}$)",
    )

    ax1.set_title(
        f"Geometric Mode Overlap ($p_i$)\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax1.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax1.set_ylabel(r"$p_i = \|\mathbf{M}_{t-1}^T \mathbf{u}_i\|^2$", fontsize=10)
    ax1.set_ylim(-0.02, 1.02)
    ax1.grid(True, linestyle="--", alpha=0.4)
    ax1.legend(frameon=True, fontsize=9, loc="upper right")

    # =========================================================================
    # Panel 2: Cumulative Energy / Power Fraction (CDF)
    # =========================================================================
    ax2.plot(
        modes,
        ht_cum_energy * 100,
        color=c_ht,
        lw=2.5,
        label=r"Heavy-Tailed ($\\\\alpha=1.2$)",
    )
    ax2.plot(
        modes,
        g_cum_energy * 100,
        color=c_gauss,
        lw=2.5,
        label=r"Gaussian ($\\\\alpha=2.0$)",
    )

    # Reference GPM threshold lines
    for thresh, style in zip([80, 90, 95], [":", "--", "-."], strict=False):
        ax2.axhline(
            thresh,
            color="gray",
            linestyle=style,
            alpha=0.6,
            lw=1.0,
            label=f"{thresh}% Threshold" if thresh == 95 else "",
        )

    # Shade the cumulative power gap
    ax2.fill_between(
        modes,
        g_cum_energy * 100,
        ht_cum_energy * 100,
        where=ht_cum_energy >= g_cum_energy,
        color=c_ht,
        alpha=0.12,
        label="Power Condensation Gain",
    )

    ax2.set_title(
        f"Cumulative Energy Fraction\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax2.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax2.set_ylabel(
        r"Cumulative Variance ($\sum_{j i} \sigma_j^2 / \sum \sigma^2$ %)", fontsize=10
    )
    ax2.set_ylim(0, 103)
    ax2.yaxis.set_major_formatter(ticker.PercentFormatter())
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend(frameon=True, fontsize=8.5, loc="lower right")

    # =========================================================================
    # Panel 3: Projected Power Contribution (w_i * p_i)
    # =========================================================================
    ax3.plot(
        modes,
        ht_proj_power * 100,
        color=c_ht,
        lw=2.2,
        marker="s",
        markersize=3.5,
        label=r"Heavy-Tailed ($\\\\alpha=1.2$)",
    )
    ax3.plot(
        modes,
        g_proj_power * 100,
        color=c_gauss,
        lw=2.2,
        marker="o",
        markersize=3.5,
        label=r"Gaussian ($\\\\alpha=2.0$)",
    )

    ax3.set_title(
        f"Projected Mode Power ($w_i \cdot p_i$)\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax3.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax3.set_ylabel("Variance Contribution to Projection (%)", fontsize=10)
    ax3.set_yscale("log")
    ax3.grid(True, linestyle="--", alpha=0.4)
    ax3.legend(frameon=True, fontsize=9, loc="upper right")

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved figure to {save_path}")

    plt.show()


if __name__ == "__main__":
    snapshot_dir = "checkpoints/mlp"
    if Path(snapshot_dir).exists():
        res_ht = run_temporal_mode_alignment_analysis(
            snapshot_dir=snapshot_dir,
            alpha=1.2,
            seed=0,
            layer_name="4",
            max_modes=150,
        )
        res_gauss = run_temporal_mode_alignment_analysis(
            snapshot_dir=snapshot_dir,
            alpha=2.0,
            seed=0,
            layer_name="4",
            max_modes=150,
        )
        plot_mode_alignment_publication(
            res_ht=res_ht,
            res_gauss=res_gauss,
            task_idx=1,
            max_modes=150,
            save_path="mode_alignment_task1.png",
        )

In [ ]:
def plot_layer_dynamics_across_tasks(
    df_metrics,
    layer_name,
    save_path=None,
    alpha_gauss=2.0,
    alpha_ht=1.2,
):
    """Plots the trajectory of Stable Rank, Effective Rank, and Spectral Entropy

    across sequential tasks (0 to T) for a specified reference layer.
    """
    # 1. Filter DataFrame for the target layer
    clean_layer_name = str(layer_name).replace(".", "_")
    sub_df = df_metrics[df_metrics["layer"] == clean_layer_name].copy()

    if sub_df.empty:
        available = df_metrics["layer"].unique()
        raise ValueError(
            f"Layer '{clean_layer_name}' not found. Available layers: {available}"
        )

    # 2. Separate Gaussian and Heavy-Tailed subsets
    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    metrics = [
        ("srank", r"Stable Rank $\operatorname{srank}(\mathbf{R})$"),
        ("erank", r"Effective Rank $\operatorname{erank}(\mathbf{R})$"),
        ("entropy", r"Spectral Entropy $S(\mathbf{R})$"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    for ax, (col_name, y_label) in zip(axes, metrics):
        # Aggregate across seeds (mean and standard error)
        g_stats = (
            df_g.groupby("task")[col_name].agg(["mean", "sem", "count"]).reset_index()
        )
        ht_stats = (
            df_ht.groupby("task")[col_name].agg(["mean", "sem", "count"]).reset_index()
        )

        # Plot Gaussian trajectory
        ax.plot(
            g_stats["task"],
            g_stats["mean"],
            marker="o",
            markersize=4,
            lw=2,
            color="#1f77b4",
            label=rf"Gaussian ($\\\\alpha={alpha_gauss}$)",
        )
        if g_stats["sem"].notna().any() and (g_stats["count"] > 1).any():
            ax.fill_between(
                g_stats["task"],
                g_stats["mean"] - g_stats["sem"],
                g_stats["mean"] + g_stats["sem"],
                color="#1f77b4",
                alpha=0.18,
            )

        # Plot Heavy-Tailed trajectory
        ax.plot(
            ht_stats["task"],
            ht_stats["mean"],
            marker="s",
            markersize=4,
            lw=2,
            color="#d62728",
            label=rf"Heavy-Tailed ($\\\\alpha={alpha_ht}$)",
        )
        if ht_stats["sem"].notna().any() and (ht_stats["count"] > 1).any():
            ax.fill_between(
                ht_stats["task"],
                ht_stats["mean"] - ht_stats["sem"],
                ht_stats["mean"] + ht_stats["sem"],
                color="#d62728",
                alpha=0.18,
            )

        ax.set_xlabel("Task Horizon ($t$)", fontsize=11)
        ax.set_ylabel(y_label, fontsize=11)
        ax.grid(True, ls="--", alpha=0.4)
        ax.legend(frameon=True, fontsize=10)

        # Format x-ticks to display integer task indices cleanly
        all_tasks = sorted(sub_df["task"].unique())
        if len(all_tasks) > 0:
            ax.set_xticks(np.arange(min(all_tasks), max(all_tasks) + 1, 2))

    fig.suptitle(
        f"Representation Variance Dynamics across Tasks — Layer: {layer_name}",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved figure to {save_path}")

    plt.show()


# Example Usage:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_layer_dynamics_across_tasks(
    df_metrics=df_metrics,
    layer_name="4",  # Middle layer key in your MLP
    # save_path="plots/dynamics_layer2_task0_to_20.pdf",
    alpha_gauss=2.0,
    alpha_ht=1.2,
)

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def parse_layer_depth(layer_name):
    """Assigns an integer depth index for sorting layers sequentially,

    ensuring the classifier/head layer is placed at the final depth.
    """
    name_str = str(layer_name).lower()

    if "classifier" in name_str or "head" in name_str or "fc_last" in name_str:
        return 9999

    digits = re.findall(r"\d+", name_str)
    if digits:
        return int(digits[0])

    return 5000


def plot_spectral_metrics_across_depth(
    df_metrics,
    task_idx=None,
    save_path=None,
    alpha_gauss=2.0,
    alpha_ht=1.2,
):
    """Plots Stable Rank, Effective Rank, and Spectral Entropy across network depth

    for a specified task (defaults to the final task in df_metrics).
    """
    # 1. Determine target task
    if task_idx is None:
        target_task = int(df_metrics["task"].max())
    else:
        target_task = int(task_idx)

    sub_df = df_metrics[df_metrics["task"] == target_task].copy()
    if sub_df.empty:
        raise ValueError(
            f"No data found for Task {target_task}. Available tasks: {sorted(df_metrics['task'].unique())}"
        )

    # 2. Sort layers by architectural depth
    sub_df["depth_order"] = sub_df["layer"].apply(parse_layer_depth)
    sub_df.sort_values(by="depth_order", inplace=True)

    ordered_layers = sub_df["layer"].unique().tolist()
    layer_display_labels = [
        "Classifier" if parse_layer_depth(l) == 9999 else f"Layer {l}"
        for l in ordered_layers
    ]
    depth_x = np.arange(len(ordered_layers))

    # 3. Filter for initialisation regimes
    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    metrics = [
        ("srank", r"Stable Rank $\operatorname{srank}(\mathbf{R})$"),
        ("erank", r"Effective Rank $\operatorname{erank}(\mathbf{R})$"),
        ("entropy", r"Spectral Entropy $S(\mathbf{R})$"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    for ax, (col_name, y_label) in zip(axes, metrics):
        # Aggregate across seeds
        g_stats = (
            df_g.groupby("layer")[col_name]
            .agg(["mean", "sem", "count"])
            .reindex(ordered_layers)
            .reset_index()
        )
        ht_stats = (
            df_ht.groupby("layer")[col_name]
            .agg(["mean", "sem", "count"])
            .reindex(ordered_layers)
            .reset_index()
        )

        # Plot Gaussian across depth
        ax.plot(
            depth_x,
            g_stats["mean"],
            marker="o",
            markersize=5,
            lw=2,
            color="#1f77b4",
            label=rf"Gaussian ($\\\\alpha={alpha_gauss}$)",
        )
        if g_stats["sem"].notna().any() and (g_stats["count"] > 1).any():
            ax.fill_between(
                depth_x,
                g_stats["mean"] - g_stats["sem"],
                g_stats["mean"] + g_stats["sem"],
                color="#1f77b4",
                alpha=0.18,
            )

        # Plot Heavy-Tailed across depth
        ax.plot(
            depth_x,
            ht_stats["mean"],
            marker="s",
            markersize=5,
            lw=2,
            color="#d62728",
            label=rf"Heavy-Tailed ($\\\\alpha={alpha_ht}$)",
        )
        if ht_stats["sem"].notna().any() and (ht_stats["count"] > 1).any():
            ax.fill_between(
                depth_x,
                ht_stats["mean"] - ht_stats["sem"],
                ht_stats["mean"] + ht_stats["sem"],
                color="#d62728",
                alpha=0.18,
            )

        ax.set_xlabel("Network Depth", fontsize=11)
        ax.set_ylabel(y_label, fontsize=11)
        ax.set_xticks(depth_x)
        ax.set_xticklabels(layer_display_labels, fontsize=10)
        ax.grid(True, ls="--", alpha=0.4)
        ax.legend(frameon=True, fontsize=10)

    fig.suptitle(
        f"Representation Variance Profiles Across Network Depth (Task {target_task})",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved depth profile figure to {save_path}")

    plt.show()


# Example Execution:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_spectral_metrics_across_depth(
    df_metrics=df_metrics,
    task_idx=20,  # Or None to auto-select the final task
    # save_path="plots/spectral_depth_task20.pdf",
    alpha_gauss=2.0,
    alpha_ht=1.2,
)

In [ ]:
import matplotlib.colors as mcolors
from scipy.interpolate import griddata


def plot_capacity_difference_heatmap(
    df_metrics,
    task_idx=1,
    min_energy=0.80,
    max_energy=0.999,
    alpha_gauss=2.0,
    alpha_ht=1.2,
    save_path=None,
    grid_resolution=(150, 150),
):
    """Generates a 2D Heatmap of Basis Savings Delta k = k_Gauss - k_HT

    across Network Depth (x-axis) vs Inverted Logarithmic Energy 1 - epsilon
    (y-axis).
    """
    # 1. Filter for the target task
    sub_df = df_metrics[df_metrics["task"] == int(task_idx)].copy()
    if sub_df.empty:
        raise ValueError(f"No records found for Task {task_idx}")

    # 2. Sort layers sequentially
    sub_df["depth_order"] = sub_df["layer"].apply(parse_layer_depth)
    sub_df.sort_values(by="depth_order", inplace=True)

    ordered_layers = sub_df["layer"].unique().tolist()
    layer_labels = [
        "Head" if parse_layer_depth(l) == 9999 else f"L{l}" for l in ordered_layers
    ]
    depth_indices = np.arange(len(ordered_layers))
    layer_to_idx = {l: i for i, l in enumerate(ordered_layers)}

    # 3. Detect energy threshold columns within [min_energy, max_energy]
    th_cols = [c for c in df_metrics.columns if c.startswith("k_")]
    valid_thresholds = []
    for c in th_cols:
        eps = parse_column_energy(c)
        if eps is not None and (min_energy <= eps <= max_energy):
            valid_thresholds.append((eps, c))

    if not valid_thresholds:
        raise ValueError(
            f"No threshold columns found in range [{min_energy}, {max_energy}]"
        )

    valid_thresholds.sort(key=lambda x: x[0])

    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    # 4. Construct point grid: Depth Index x Inverted Log Remaining Energy log10(1 - eps)
    points_x, points_y, delta_k_vals = [], [], []

    for eps, col_key in valid_thresholds:
        g_means = df_g.groupby("layer")[col_key].mean().reindex(ordered_layers)
        ht_means = df_ht.groupby("layer")[col_key].mean().reindex(ordered_layers)
        delta_k = (g_means - ht_means).to_numpy()

        y_val = 1.0 - eps  # Remaining energy fraction
        for d_idx, d_k in zip(depth_indices, delta_k):
            points_x.append(d_idx)
            points_y.append(y_val)
            delta_k_vals.append(d_k)

    points_x = np.array(points_x)
    points_y = np.array(points_y)
    delta_k_vals = np.array(delta_k_vals)

    # 5. Continuous 2D grid interpolation
    grid_x, grid_y = np.meshgrid(
        np.linspace(depth_indices.min(), depth_indices.max(), grid_resolution[0]),
        np.geomspace(points_y.min(), points_y.max(), grid_resolution[1]),
    )

    # Interpolate in log-space for smooth vertical transitions
    grid_z = griddata(
        (points_x, np.log10(points_y)),
        delta_k_vals,
        (grid_x, np.log10(grid_y)),
        method="cubic",
    )

    # 6. Plotting
    fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
    ax.set_box_aspect(1)

    # Symmetrical colormap range centered at Delta k = 0
    max_abs_delta = np.nanmax(np.abs(delta_k_vals))
    norm = mcolors.TwoSlopeNorm(vcenter=0.0, vmin=-max_abs_delta, vmax=max_abs_delta)

    mesh = ax.pcolormesh(
        grid_x,
        grid_y,
        grid_z,
        shading="gouraud",
        cmap="seismic",  # Blue = Heavy-Tailed Savings (>0), Red = Gaussian Lower (<0)
        norm=norm,
    )

    # Contour lines showing boundary of zero difference
    ax.contour(
        grid_x,
        grid_y,
        grid_z,
        levels=[0.0],
        colors="black",
        linewidths=1.2,
        linestyles="--",
    )

    # Configure Inverted Logarithmic Y-Axis
    ax.set_yscale("log")
    ax.invert_yaxis()  # Top of plot = high energy (99.9%), Bottom = lower energy (80%)

    # Custom tick positions for intuitive percentage readings
    target_ticks_pct = [0.80, 0.90, 0.95, 0.98, 0.99, 0.995, 0.999]
    actual_ticks = [
        1.0 - p
        for p in target_ticks_pct
        if points_y.min() <= (1.0 - p) <= points_y.max()
    ]
    ax.yaxis.set_major_locator(ticker.FixedLocator(actual_ticks))
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda y, _: f"{((1.0 - y) * 100):.1f}%")
    )
    ax.yaxis.set_minor_locator(ticker.NullLocator())

    ax.set_xticks(depth_indices)
    ax.set_xticklabels(layer_labels, fontsize=10.5)

    ax.set_xlabel("Network Depth", fontsize=11, labelpad=8)
    ax.set_ylabel(
        r"Energy Threshold ($\epsilon$, expanded log tail)",
        fontsize=11,
        labelpad=8,
    )
    ax.set_title(
        f"GPM Capacity Delta Map: $\Delta k_\epsilon = k_{{\mathrm{{Gauss}}}} - k_{{\mathrm{{HT}}}}$ (Task {task_idx})",
        fontsize=12,
        fontweight="semibold",
        pad=12,
    )

    cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(
        r"Basis Vectors Saved by Heavy Tails ($\Delta k_\epsilon$)",
        fontsize=10.5,
        labelpad=8,
    )

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved capacity delta heatmap to {save_path}")

    plt.show()


# Example Execution:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_capacity_difference_heatmap(
    df_metrics=df_metrics,
    task_idx=1,
    min_energy=0.80,
    max_energy=0.999,
    # save_path="plots/capacity_delta_heatmap_task1.pdf"
)

In [ ]:
def get_ordered_layer_columns(df):
    """Detects and sorts total/projected variance column pairs by layer depth."""
    total_cols = [c for c in df.columns if c.startswith("var_total_")]
    layer_keys = [c.replace("var_total_", "") for c in total_cols]
    layer_keys.sort(key=parse_layer_depth)

    display_labels = [
        "Head" if parse_layer_depth(k) == 9999 else f"L{parse_layer_depth(k) // 2}"
        for k in layer_keys
    ]
    return layer_keys, display_labels


def plot_representation_projection_energy(
    csv_ht_path="gpm_a1.2_run_s0.csv",
    csv_gauss_path="gpm_a2.0_run_s0.csv",
    depth_task_idx=1,
    mid_layer_key="features_8_weight",
    save_path=None,
):
    """Plots Representation Projection Energy Ratio (E_proj = var_proj / var_total)

    across both the temporal task axis and the depth axis.
    """
    # 1. Load and clean task completion rows
    df_ht = pd.read_csv(csv_ht_path).dropna(subset=["task_idx"]).sort_values("task_idx")
    df_g = (
        pd.read_csv(csv_gauss_path).dropna(subset=["task_idx"]).sort_values("task_idx")
    )

    layer_keys, layer_display_labels = get_ordered_layer_columns(df_ht)

    # 2. Extract Temporal Dynamics (Exclude task 0 where basis is uninitialized)
    df_ht_temp = df_ht[df_ht["task_idx"] >= 1].copy()
    df_g_temp = df_g[df_g["task_idx"] >= 1].copy()

    tasks_temp = df_ht_temp["task_idx"].astype(int).to_numpy()

    ht_temp_proj = (
        df_ht_temp[f"var_proj_{mid_layer_key}"]
        / df_ht_temp[f"var_total_{mid_layer_key}"]
    ).to_numpy()
    g_temp_proj = (
        df_g_temp[f"var_proj_{mid_layer_key}"] / df_g_temp[f"var_total_{mid_layer_key}"]
    ).to_numpy()

    # 3. Extract Depth Profile for Selected Task
    row_ht_task = df_ht[df_ht["task_idx"] == depth_task_idx]
    row_g_task = df_g[df_g["task_idx"] == depth_task_idx]

    if row_ht_task.empty or row_g_task.empty:
        raise ValueError(f"Task {depth_task_idx} not found in one or both CSV files.")

    ht_depth_proj = np.array(
        [
            (row_ht_task[f"var_proj_{k}"] / row_ht_task[f"var_total_{k}"]).iloc[0]
            for k in layer_keys
        ]
    )
    g_depth_proj = np.array(
        [
            (row_g_task[f"var_proj_{k}"] / row_g_task[f"var_total_{k}"]).iloc[0]
            for k in layer_keys
        ]
    )

    depth_x = np.arange(len(layer_keys))
    x_dense = np.linspace(depth_x.min(), depth_x.max(), 300)

    # Smooth splines for depth profile
    spline_ht = make_interp_spline(depth_x, ht_depth_proj, k=3)
    spline_g = make_interp_spline(depth_x, g_depth_proj, k=3)

    # 4. Canvas Setup: Side-by-Side Panels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

    color_gauss = "#d62728"
    color_ht = "#1f77b4"

    # --- Panel A: Temporal Evolution ---
    ax1.plot(
        tasks_temp,
        g_temp_proj,
        marker="o",
        markersize=5,
        color=color_gauss,
        lw=2.2,
        label=r"Gaussian ($\\\\alpha=2.0$)",
    )
    ax1.plot(
        tasks_temp,
        ht_temp_proj,
        marker="s",
        markersize=5,
        color=color_ht,
        lw=2.2,
        label=r"Heavy-Tailed ($\\\\alpha=1.2$)",
    )

    ax1.fill_between(
        tasks_temp,
        g_temp_proj,
        ht_temp_proj,
        where=ht_temp_proj >= g_temp_proj,
        color=color_ht,
        alpha=0.14,
        label=r"Subspace Alignment Gain ($\Delta E_{\mathrm{proj}}$)",
    )

    ax1.set_title(
        f"Temporal Subspace Recycling ({layer_display_labels[layer_keys.index(mid_layer_key)]})",
        fontsize=12,
        fontweight="semibold",
        pad=10,
    )
    ax1.set_xlabel("Task Index ($t$)", fontsize=11, labelpad=8)
    ax1.set_ylabel(
        r"Projection Energy Ratio $E_{\mathrm{proj}} = \frac{\|\mathbf{M}_{t-1}^T \mathbf{R}_t\|_F^2}{\|\mathbf{R}_t\|_F^2}$",
        fontsize=11,
        labelpad=8,
    )
    ax1.set_xticks(tasks_temp[::2])
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y * 100:.0f}%"))
    ax1.grid(True, linestyle="--", alpha=0.35)
    ax1.legend(frameon=True, fontsize=9.5, loc="lower right")

    # --- Panel B: Depth Profile ---
    ax2.plot(
        x_dense,
        np.clip(spline_g(x_dense), 0, 1),
        color=color_gauss,
        lw=2.2,
        label=r"Gaussian ($\\\\alpha=2.0$)",
    )
    ax2.scatter(depth_x, g_depth_proj, color=color_gauss, s=32, zorder=5)

    ax2.plot(
        x_dense,
        np.clip(spline_ht(x_dense), 0, 1),
        color=color_ht,
        lw=2.2,
        label=r"Heavy-Tailed ($\\\\alpha=1.2$)",
    )
    ax2.scatter(depth_x, ht_depth_proj, color=color_ht, s=32, marker="s", zorder=5)

    ax2.fill_between(
        x_dense,
        np.clip(spline_g(x_dense), 0, 1),
        np.clip(spline_ht(x_dense), 0, 1),
        where=spline_ht(x_dense) >= spline_g(x_dense),
        color=color_ht,
        alpha=0.14,
        label=r"Subspace Alignment Gain ($\Delta E_{\mathrm{proj}}$)",
    )

    ax2.set_title(
        f"Depth-Wise Subspace Overlap (Task {depth_task_idx})",
        fontsize=12,
        fontweight="semibold",
        pad=10,
    )
    ax2.set_xlabel("Network Depth", fontsize=11, labelpad=8)
    ax2.set_ylabel(
        r"Projection Energy Ratio $E_{\mathrm{proj}}$", fontsize=11, labelpad=8
    )
    ax2.set_xticks(depth_x)
    ax2.set_xticklabels(layer_display_labels, fontsize=10)
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y * 100:.0f}%"))
    ax2.grid(True, linestyle="--", alpha=0.35)
    ax2.legend(frameon=True, fontsize=9.5, loc="lower right")

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved alignment figure to {save_path}")

    plt.show()


# Example Execution:
plot_representation_projection_energy(
    csv_ht_path="gpm_a1.2_run_s0.csv",
    csv_gauss_path="gpm_a2.0_run_s0.csv",
    depth_task_idx=1,
    mid_layer_key="features_8_weight",
    save_path="plots/representation_projection_energy.pdf",
)

In [ ]:
# --- 1. CONFIGURATION & PATHS ---
RESULTS_DIR = Path("./phase_sweep/results")

# Match the exact grid parameters used in your sweep
ALPHA_VALS = np.round(np.arange(1.0, 2.01, 0.1), 2)  # 1.0 to 2.0 (11 steps)
G_VALS = np.round(np.arange(0.5, 3.01, 0.25), 2)  # 0.5 to 3.0 (11 steps)

NUM_ALPHA = len(ALPHA_VALS)
NUM_G = len(G_VALS)

# Mapping indices for array insertion
alpha_to_idx = {a: i for i, a in enumerate(ALPHA_VALS)}
g_to_idx = {g: j for j, g in enumerate(G_VALS)}

# Matrices configured for FLIPPED AXES: Rows = g (vertical), Columns = alpha (horizontal)
matrix_final_acc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_task20_auc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_hidden_rank = np.full((NUM_G, NUM_ALPHA), np.nan)


# --- 2. DATA AGGREGATION MATCHING YOUR CSV HEADERS ---
if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found at {RESULTS_DIR}")

csv_files = list(RESULTS_DIR.glob("results_a*_g*.csv"))
print(f"Found {len(csv_files)} result CSV files. Parsing matching headers...")

for csv_path in csv_files:
    # Parse alpha and g from filename (e.g., results_a1.20_g1.50.csv)
    filename = csv_path.stem
    parts = filename.split("_")
    alpha_val = float(parts[1].replace("a", ""))
    g_val = float(parts[2].replace("g", ""))

    if alpha_val not in alpha_to_idx or g_val not in g_to_idx:
        continue

    # Note the flipped index order: row = g, col = alpha
    row_g = g_to_idx[g_val]
    col_a = alpha_to_idx[alpha_val]

    df = pd.read_csv(csv_path)

    # Match exact accuracy columns: task_0_acc ... task_19_acc
    acc_cols = [f"task_{t}_acc" for t in range(20) if f"task_{t}_acc" in df.columns]

    # A. Final Average Accuracy (Mean across all 20 tasks in the absolute final state)
    if acc_cols:
        final_row = df.iloc[-1]
        matrix_final_acc[row_g, col_a] = final_row[acc_cols].mean()

    # B. Final Task (Task 19) Plasticity / Trajectory AUC
    # Measures the normalized Area Under the Curve (AUC) for Task 19 during its training phase
    if "task_19_acc" in df.columns:
        t20_values = df["task_19_acc"].dropna()
        if len(t20_values) > 0:
            # Take the final evaluated accuracy score for Task 20
            matrix_task20_auc[row_g, col_a] = t20_values.iloc[-1]

    # C. Hidden Layer Reserved Rank (Excludes Layer 1: basis_rank_features.0.weight)
    if (
        "total_basis_rank" in df.columns
        and "basis_rank_features.0.weight" in df.columns
    ):
        final_total_rank = df["total_basis_rank"].iloc[-1]
        final_layer1_rank = df["basis_rank_features.0.weight"].iloc[-1]
        matrix_hidden_rank[row_g, col_a] = final_total_rank - final_layer1_rank
    elif "total_basis_rank" in df.columns:
        matrix_hidden_rank[row_g, col_a] = df["total_basis_rank"].iloc[-1]

print("Grid aggregation complete!")


# --- 3. FLIPPED AXES PHASE DIAGRAM PLOTTING ROUTINE ---
def plot_phase_heatmap(matrix_data, title, cbar_label, cmap="viridis", fmt=".2f"):
    plt.figure(figsize=(10, 7.5))

    # Heatmap setup: Y-axis = g (vertical), X-axis = alpha (horizontal)
    ax = sns.heatmap(
        matrix_data,
        xticklabels=ALPHA_VALS,
        yticklabels=G_VALS,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": cbar_label},
        annot_kws={"size": 8.5, "weight": "bold"},
    )

    # Invert Y-axis so g increases upwards (standard physics convention)
    ax.invert_yaxis()

    plt.xlabel(r"Tail Exponent ($\\\\alpha$)", fontweight="bold", fontsize=12)
    plt.ylabel(r"Initialization Gain ($g$)", fontweight="bold", fontsize=12)
    plt.title(title, fontweight="bold", fontsize=13, pad=14)

    plt.tight_layout()
    plt.show()


# --- 4. GENERATE THE THREE PHASE DIAGRAMS ---

# 1. Final Average Accuracy Phase Diagram
plot_phase_heatmap(
    matrix_final_acc,
    title="Phase Diagram: Final 20-Task Average Test Accuracy\n"
    "Maps Continual Learning Performance Across Parameter Space",
    cbar_label="Mean Test Accuracy",
    cmap="magma",
    fmt=".3f",
)

# 2. Final Task (Task 20) Plasticity / Trajectory AUC Phase Diagram
plot_phase_heatmap(
    matrix_task20_auc,
    title="Phase Diagram: Task 20 Accuracy\n"
    "Identifies Capacity Exhaustion vs. Retained Learning Ability",
    cbar_label="Task 20 Final Accuracy",
    cmap="plasma",
    fmt=".3f",
)

# 3. Hidden Layer Reserved Basis Rank Phase Diagram (Excludes Layer 1)
plot_phase_heatmap(
    matrix_hidden_rank,
    title="Phase Diagram: Hidden Layer Reserved Basis Rank ($K_{\\text{hidden}}$)\n"
    "Quantifies Subspace Compression (Excludes Invariant Layer 1)",
    cbar_label="Hidden Layer Basis Vectors",
    cmap="viridis_r",  # Reversed so lower rank (higher compression) stands out
    fmt=".0f",
)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- 1. CONFIGURATION ---
GAUSSIAN_CSV = "gpm_a2.0_run_s0.csv"  # Standard Gaussian initialization run
HEAVY_TAIL_CSV = "gpm_a1.2_run_s0.csv"  # Heavy-Tailed initialization run
NUM_TASKS = 20


# --- 2. EXTRACTOR FUNCTION FOR A SINGLE CSV FILE ---
def extract_run_metrics(filepath, num_tasks=20):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find target CSV file at: '{filepath}'")

    df = pd.read_csv(filepath)
    num_entries = len(df)

    # Sort chronological accuracy columns
    task_acc_cols = [
        c for c in df.columns if c.startswith("task_") and c.endswith("_acc")
    ]
    task_acc_cols = sorted(task_acc_cols, key=lambda x: int(x.split("_")[1]))

    # Determine active task footprint per row to find task completion boundaries
    active_tasks_per_step = np.array(
        [df.iloc[idx][task_acc_cols].notna().sum() for idx in range(num_entries)]
    )

    # Detect transition step indices where each task finishes training
    transition_indices = []
    current_num = active_tasks_per_step[0]
    for idx in range(1, num_entries):
        if active_tasks_per_step[idx] > current_num:
            transition_indices.append(idx - 1)
            current_num = active_tasks_per_step[idx]
    transition_indices.append(num_entries - 1)

    if len(transition_indices) != num_tasks:
        print(
            f"Warning: Boundary mismatch in {filepath}. Detected {len(transition_indices)} boundaries instead of {num_tasks}"
        )

    # Determine the step intervals for each task
    task_step_intervals = []
    start_idx = 0
    for end_idx in transition_indices:
        task_step_intervals.append((start_idx, end_idx))
        start_idx = end_idx + 1

    # Storage arrays
    avg_acc_at_wrapup = []
    current_task_auc = []
    total_basis_rank = []
    layer_1_basis_rank = []

    # Detect available rank columns dynamically
    layer_1_col = "basis_rank_0" if "basis_rank_0" in df.columns else None
    if layer_1_col is None:
        layer_1_cols = [
            c
            for c in df.columns
            if "basis" in c and ("0" in c or "layer_0" in c or "fc1" in c)
        ]
        layer_1_col = layer_1_cols[0] if len(layer_1_cols) > 0 else None

    for t_idx, boundary_idx in enumerate(transition_indices):
        # 1. Global Average Accuracy across all active tasks
        row_accs = df.iloc[boundary_idx][task_acc_cols].values
        active_accs = row_accs[~pd.isna(row_accs)]
        avg_acc_at_wrapup.append(np.mean(active_accs) if len(active_accs) > 0 else 0.0)

        # 2. Plasticity AUC during training of Task t_idx
        start_step_idx, end_step_idx = task_step_intervals[t_idx]
        task_col = f"task_{t_idx}_acc"

        task_trajectory = (
            df.iloc[start_step_idx : end_step_idx + 1][task_col].dropna().values
        )
        steps = (
            df.iloc[start_step_idx : end_step_idx + 1]["step"]
            .iloc[: len(task_trajectory)]
            .values
        )

        if len(task_trajectory) > 1:
            auc_val = np.trapezoid(y=task_trajectory, x=steps) / (steps[-1] - steps[0])
        elif len(task_trajectory) == 1:
            auc_val = task_trajectory[0]
        else:
            auc_val = 0.0
        current_task_auc.append(auc_val)

        # 3. Total Basis Rank at task wrap-up
        if "total_basis_rank" in df.columns:
            total_rank_val = df.iloc[boundary_idx]["total_basis_rank"]
        else:
            rank_cols = [c for c in df.columns if c.startswith("basis_rank_")]
            total_rank_val = (
                df.iloc[boundary_idx][rank_cols].sum() if len(rank_cols) > 0 else np.nan
            )
        total_basis_rank.append(total_rank_val)

        # 4. Layer 1 Basis Rank at task wrap-up
        if layer_1_col and layer_1_col in df.columns:
            layer_1_val = df.iloc[boundary_idx][layer_1_col]
        else:
            layer_1_val = np.nan
        layer_1_basis_rank.append(layer_1_val)

    return {
        "avg_acc": avg_acc_at_wrapup,
        "auc": current_task_auc,
        "total_rank": total_basis_rank,
        "layer1_rank": layer_1_basis_rank,
    }


# --- 3. PROCESS BOTH RUNS ---
print(f"Extracting Gaussian baseline metrics from: {GAUSSIAN_CSV}")
gauss_data = extract_run_metrics(GAUSSIAN_CSV, NUM_TASKS)

print(f"Extracting Heavy-Tailed metrics from: {HEAVY_TAIL_CSV}")
ht_data = extract_run_metrics(HEAVY_TAIL_CSV, NUM_TASKS)


# --- 4. 2x2 COMPARATIVE VISUALIZATION ---
fig, axs = plt.subplots(2, 2, figsize=(15, 11), dpi=100)
tasks_axis = np.arange(1, NUM_TASKS + 1)

# Color and style definitions
gauss_color, ht_color = "crimson", "dodgerblue"
gauss_marker, ht_marker = "o", "s"

# [TOP LEFT] Global Cumulative Average Accuracy
axs[0, 0].plot(
    tasks_axis,
    gauss_data["avg_acc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 0].plot(
    tasks_axis,
    ht_data["avg_acc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 0].set_title("1. Global Average Test Accuracy", fontsize=11, weight="bold")
axs[0, 0].set_xlabel("Task Index")
axs[0, 0].set_ylabel("Mean Accuracy across Learned Tasks")
axs[0, 0].set_xticks(tasks_axis)
axs[0, 0].grid(True, linestyle=":", alpha=0.5)
axs[0, 0].legend()

# [TOP RIGHT] Task Plasticity AUC
axs[0, 1].plot(
    tasks_axis,
    gauss_data["auc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 1].plot(
    tasks_axis,
    ht_data["auc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 1].set_title(
    "2. Current Task Learning Curve AUC (Plasticity)", fontsize=11, weight="bold"
)
axs[0, 1].set_xlabel("Task Index")
axs[0, 1].set_ylabel("Normalized Training AUC")
axs[0, 1].set_xticks(tasks_axis)
axs[0, 1].grid(True, linestyle=":", alpha=0.5)
axs[0, 1].legend()

# [BOTTOM LEFT] Total Basis Rank
axs[1, 0].plot(
    tasks_axis,
    gauss_data["total_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 0].plot(
    tasks_axis,
    ht_data["total_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 0].set_title("3. Cumulative Total Basis Rank", fontsize=11, weight="bold")
axs[1, 0].set_xlabel("Task Index")
axs[1, 0].set_ylabel("Total Reserved Basis Vectors")
axs[1, 0].set_xticks(tasks_axis)
axs[1, 0].grid(True, linestyle=":", alpha=0.5)
axs[1, 0].legend()

# [BOTTOM RIGHT] Layer 1 Basis Rank
axs[1, 1].plot(
    tasks_axis,
    gauss_data["layer1_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 1].plot(
    tasks_axis,
    ht_data["layer1_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 1].set_title("4. Layer 1 Basis Rank", fontsize=11, weight="bold")
axs[1, 1].set_xlabel("Task Index")
axs[1, 1].set_ylabel("Layer 1 Reserved Basis Vectors")
axs[1, 1].set_xticks(tasks_axis)
axs[1, 1].grid(True, linestyle=":", alpha=0.5)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig("gpm_gaussian_vs_heavytail_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms

sys.path.append(os.path.abspath(".."))
from src.architectures import GeneralMLP
from src.utils import set_seed


def get_gpu_data(dataset):
    """Loads raw tensors to GPU once to avoid repetitive overhead."""
    imgs = torch.stack([img for img, _ in dataset]).to(DEVICE).view(-1, 784)
    lbls = torch.tensor([lbl for _, lbl in dataset]).to(DEVICE)
    return imgs, lbls


def generate_permutations(num_tasks, num_pixels=784, seed=42):
    """Generates clean pseudorandom domain permutations mapping each task stream."""
    rng = np.random.RandomState(seed)
    perms = [
        torch.arange(num_pixels).to(DEVICE)
    ]  # Task 0 is standard un-permuted MNIST
    for _ in range(num_tasks - 1):
        perms.append(torch.from_numpy(rng.permutation(num_pixels)).to(DEVICE))
    return perms


# --- 1. CONFIGURATION & PATHS ---
PATH_GAUSSIAN = Path("./checkpoints/mlp/snapshot_A2.0_T20_s0.pt")
PATH_HEAVY_TAILED = Path("./checkpoints/mlp/snapshot_A1.2_T20_s0.pt")

NUM_TASKS = 20
BATCH_SIZE = 1024  # Size of the evaluation batch per task

# Architecture hyper-parameters
HIDDEN_SIZE = 784
DEPTH = 9
ACTIVATION_NAME = "tanh"
BIAS = False
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 2. EXTRACTION HELPER FUNCTION ---
def extract_model_and_activations(
    snapshot_path: Path,
    test_imgs_raw: torch.Tensor,
    num_tasks: int = 20,
    batch_size: int = 1024,
    device: torch.device = DEVICE,
):
    """Loads a snapshot checkpoint, reconstructs the model, and extracts

    both pre-activations (h) and post-activations (x = phi(h)) across all tasks.
    """
    if not snapshot_path.exists():
        raise FileNotFoundError(f"Snapshot not found at: {snapshot_path}")

    print(f"\n--- Processing Snapshot: {snapshot_path.name} ---")
    snapshot = torch.load(snapshot_path, map_location=device, weights_only=False)
    metadata = snapshot.get("metadata", {})
    seed = metadata.get("seed", 0)

    print(
        f"Metadata -> Task: {metadata.get('task')}, Epoch: {metadata.get('epoch')}, Seed: {seed}"
    )

    # Reconstruct architecture and load weights
    model = GeneralMLP(
        input_size=784,
        hidden_size=HIDDEN_SIZE,
        num_classes=10,
        depth=DEPTH,
        activation_name=ACTIVATION_NAME,
        bias=BIAS,
    ).to(device)

    model.load_state_dict(snapshot["state_dict"])
    model.eval()

    # Resolve activation function for post-activations
    act_lookup = {
        "tanh": torch.tanh,
        "relu": torch.relu,
        "sigmoid": torch.sigmoid,
    }
    act_fn = act_lookup.get(ACTIVATION_NAME.lower(), torch.tanh)

    # Regenerate task permutations matching the snapshot's seed
    set_seed(SEED)
    task_permutations = generate_permutations(num_tasks=num_tasks, seed=SEED)

    # Extract pre- and post-activations per layer per task
    task_pre_acts = []
    task_post_acts = []

    with torch.no_grad():
        for t_idx in range(num_tasks):
            perm = task_permutations[t_idx]
            task_batch = test_imgs_raw[:batch_size, perm]

            # 1. Extract Pre-activations h
            pre_acts = model.get_pre_activations(task_batch)

            if isinstance(pre_acts, dict):
                layer_keys = [k for k in pre_acts]
                pre_list = [pre_acts[k] for k in layer_keys]
            else:
                pre_list = pre_acts

            # 2. Derive Post-activations x = phi(h)
            # Hidden layers pass through phi; final readout/logits remain linear
            post_list = []
            num_layers = len(pre_list)
            for l_idx, h in enumerate(pre_list):
                if l_idx < num_layers - 1:
                    post_list.append(act_fn(h).detach())
                else:
                    post_list.append(
                        h.detach()
                    )  # Unactivated logits for the readout head

            task_pre_acts.append(pre_list)
            task_post_acts.append(post_list)

    print(f"Extraction complete for {snapshot_path.name}.")

    return {
        "model": model,
        "task_pre_acts": task_pre_acts,
        "task_post_acts": task_post_acts,
        "metadata": metadata,
        "seed": seed,
    }


# --- 3. MAIN EXECUTION PIPELINE ---

# A. Prepare Data (Loaded once for both runs)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)
test_imgs_raw, _ = get_gpu_data(mnist_test)  # Unpermuted GPU images [N, 784]

# B. Process Both Snapshots
run_gaussian = extract_model_and_activations(
    snapshot_path=PATH_GAUSSIAN,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

run_ht = extract_model_and_activations(
    snapshot_path=PATH_HEAVY_TAILED,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

print("\nExtraction successfully completed for both Gaussian and Heavy-Tailed runs!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# --- CONFIGURATION ---
TARGET_TASK = 0  # Task index to visualize (0 to 19)
TARGET_LAYER = 4  # Hidden layer depth (e.g., intermediate layer 4)
NUM_SAMPLES = 200  # Number of samples to show in heatmap

# 1. Extract post-activations: Shape [Batch, Neurons]
act_gauss = run_gaussian["task_post_acts"][TARGET_TASK][TARGET_LAYER].cpu().numpy()
act_ht = run_ht["task_post_acts"][TARGET_TASK][TARGET_LAYER].cpu().numpy()

# 2. Compute mean absolute activity and variance per neuron
mean_act_gauss = np.mean(np.abs(act_gauss), axis=0)
mean_act_ht = np.mean(np.abs(act_ht), axis=0)

# --- PLOTTING ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=150)

# Panel A: Spatial Coordinate Sparsity Profile
axes[0, 0].plot(mean_act_gauss, color="#1f77b4", lw=1.0, alpha=0.85)
axes[0, 0].set_title(
    f"Gaussian (α=2.0) | Layer {TARGET_LAYER} Neuron Activity",
    fontsize=11,
    fontweight="bold",
)
axes[0, 0].set_ylabel(r"Mean Absolute Activation $\mathbb{E}[|x_i|]$")
axes[0, 0].set_ylim(0, 1.1)
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

axes[0, 1].plot(mean_act_ht, color="#d62728", lw=1.0, alpha=0.85)
axes[0, 1].set_title(
    f"Heavy-Tailed (α=1.2) | Layer {TARGET_LAYER} Neuron Activity",
    fontsize=11,
    fontweight="bold",
)
axes[0, 1].set_ylim(0, 1.1)
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# Panel B: Raw Activation Heatmaps [Samples x Neurons]
im0 = axes[1, 0].imshow(
    act_gauss[:NUM_SAMPLES],
    aspect="auto",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)
axes[1, 0].set_title("Gaussian: Diffuse & Uniform Manifold", fontsize=11)
axes[1, 0].set_xlabel("Neuron Index (1 to 784)")
axes[1, 0].set_ylabel("Sample Index")

im1 = axes[1, 1].imshow(
    act_ht[:NUM_SAMPLES],
    aspect="auto",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)
axes[1, 1].set_title("Heavy-Tailed: Spiky & Sparse Localization", fontsize=11)
axes[1, 1].set_xlabel("Neuron Index (1 to 784)")

# Formatting colorbar
cbar = fig.colorbar(
    im1,
    ax=axes[1, :],
    orientation="horizontal",
    fraction=0.05,
    pad=0.2,
    shrink=0.6,
)
cbar.set_label(r"Post-Activation Value $\tanh(h_i)$")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn


def compute_cumulative_jacobians(
    run_data: dict, activation_name: str = "tanh", compute_svd: bool = True
) -> list:
    """Computes cumulative post-activation Jacobians J_{0 -> l} across depth

    under the mean-field approximation.
    """
    model = run_data["model"]
    task_pre_acts = run_data["task_pre_acts"]
    num_tasks = len(task_pre_acts)

    # Extract all Linear layer weight tensors
    weights = [
        m.weight.detach().to(torch.float32)
        for m in model.modules()
        if isinstance(m, nn.Linear)
    ]
    num_layers = len(weights)

    def get_activation_derivative(h: torch.Tensor, act_name: str) -> torch.Tensor:
        act = act_name.lower()
        if act == "tanh":
            return 1.0 - torch.tanh(h) ** 2
        elif act == "relu":
            return (h > 0.0).to(h.dtype)
        elif act == "sigmoid":
            s = torch.sigmoid(h)
            return s * (1.0 - s)
        else:
            return torch.ones_like(h)

    cumulative_jacobians = []

    for t_idx in range(num_tasks):
        pre_acts_t = task_pre_acts[t_idx]
        task_results = []
        J_cum = None

        for l_idx in range(num_layers):
            W_l = weights[l_idx]  # Shape: [d_out, d_in]
            h_l = pre_acts_t[l_idx].to(
                device=W_l.device, dtype=W_l.dtype
            )  # Ensure device/dtype match

            # 1. Mean-Field Activation Derivative D^l
            if l_idx < num_layers - 1:
                d_l = get_activation_derivative(h_l, activation_name).mean(
                    dim=0
                )  # [d_out]
            else:
                d_l = torch.ones(W_l.shape[0], device=W_l.device, dtype=W_l.dtype)

            # 2. Layerwise Post-Activation Jacobian: J_post^l = D^l @ W^l
            J_post_l = d_l.unsqueeze(1) * W_l  # Shape: [d_out, d_in]

            # 3. Chain Cumulative Operator: J_{0 -> l} = J_post^l @ J_{0 -> l-1}
            J_cum = J_post_l if J_cum is None else (J_post_l @ J_cum)

            layer_dict = {
                "layer_idx": l_idx,
                "J_cum": J_cum.clone(),
                "shape": tuple(J_cum.shape),
            }

            # 4. Singular Value Decomposition
            if compute_svd:
                U, S, Vh = torch.linalg.svd(J_cum, full_matrices=False)
                layer_dict.update(
                    {
                        "U": U.detach(),
                        "S": S.detach(),
                        "Vh": Vh.detach(),
                    }
                )

            task_results.append(layer_dict)

        cumulative_jacobians.append(task_results)

    return cumulative_jacobians


# Process cumulative Jacobians for both experimental runs
jacobians_gaussian = compute_cumulative_jacobians(
    run_gaussian, activation_name="tanh", compute_svd=True
)
jacobians_ht = compute_cumulative_jacobians(
    run_ht, activation_name="tanh", compute_svd=True
)

# Inspect Layer 4 cumulative Jacobian at Task 0
layer_4_ht = jacobians_ht[0][4]
print(f"Task 0, Layer 4 Cumulative Operator Shape: {layer_4_ht['shape']}")
print(f"Top 5 Singular Values: {layer_4_ht['S'][:5].cpu().numpy()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# --- CONFIGURATION ---
TARGET_TASK = 0  # Task index to analyze (e.g. Task 0)
SELECTED_LAYERS = [4]  # 0-indexed hidden layers (Layers 1, 3, 5, 8)
FOCAL_LAYER = 4  # Layer to highlight for the 97% GPM threshold panel
GPM_THRESHOLD = 0.97

# Color palette across depth
layer_colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=150)

# =========================================================================
# PANEL 1: Log-Log Singular Value Spectra across Depth
# =========================================================================
for idx, l_idx in enumerate(SELECTED_LAYERS):
    # Extract singular values [min(d_out, d_in)]
    s_gauss = jacobians_gaussian[TARGET_TASK][l_idx]["S"].cpu().numpy().astype(float)
    s_ht = jacobians_ht[TARGET_TASK][l_idx]["S"].cpu().numpy().astype(float)

    # Normalize by top singular value for clean scale-free comparison
    s_gauss_norm = s_gauss / (s_gauss[0] + 1e-12)
    s_ht_norm = s_ht / (s_ht[0] + 1e-12)

    k_modes = np.arange(1, len(s_gauss) + 1)
    col = layer_colors[idx % len(layer_colors)]

    # Gaussian: Dashed lines | Heavy-Tailed: Solid lines
    axes[0].plot(
        k_modes,
        s_gauss_norm,
        label=f"Gauss (L{l_idx + 1})",
        color=col,
        linestyle="--",
        lw=1.5,
        alpha=0.75,
    )
    axes[0].plot(
        k_modes,
        s_ht_norm,
        label=f"HT (L{l_idx + 1})",
        color=col,
        linestyle="-",
        lw=2.2,
    )

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title(
    r"Log-Log Singular Spectra of $\mathbf{J}_{0 \to l}$",
    fontsize=12,
    fontweight="bold",
)
axes[0].set_xlabel("Mode Index $k$", fontsize=11)
axes[0].set_ylabel(r"Normalized Singular Value $\sigma_k / \sigma_1$", fontsize=11)
axes[0].grid(True, which="both", linestyle="--", alpha=0.4)
axes[0].legend(ncol=2, fontsize=9, loc="lower left", framealpha=0.9)

# =========================================================================
# PANEL 2: Cumulative Variance Explained & GPM Threshold Crossing
# =========================================================================
s_g_focal = (
    jacobians_gaussian[TARGET_TASK][FOCAL_LAYER]["S"].cpu().numpy().astype(float)
)
s_h_focal = jacobians_ht[TARGET_TASK][FOCAL_LAYER]["S"].cpu().numpy().astype(float)

cum_var_g = np.cumsum(s_g_focal**2) / np.sum(s_g_focal**2)
cum_var_h = np.cumsum(s_h_focal**2) / np.sum(s_h_focal**2)
k_modes_focal = np.arange(1, len(s_g_focal) + 1)

axes[1].plot(
    k_modes_focal,
    cum_var_g * 100,
    label=r"Gaussian ($\\\\alpha=2.0$)",
    color="#1f77b4",
    lw=2.2,
    linestyle="--",
)
axes[1].plot(
    k_modes_focal,
    cum_var_h * 100,
    label=r"Heavy-Tailed ($\\\\alpha=1.2$)",
    color="#d62728",
    lw=2.2,
)
axes[1].axhline(
    GPM_THRESHOLD * 100,
    color="black",
    linestyle=":",
    lw=1.5,
    label=f"{int(GPM_THRESHOLD * 100)}% GPM Basis Threshold",
)

# Identify exact crossing ranks (K)
k_g_cross = (
    np.where(cum_var_g >= GPM_THRESHOLD)[0][0] + 1
    if np.any(cum_var_g >= GPM_THRESHOLD)
    else len(s_g_focal)
)
k_h_cross = (
    np.where(cum_var_h >= GPM_THRESHOLD)[0][0] + 1
    if np.any(cum_var_h >= GPM_THRESHOLD)
    else len(s_h_focal)
)

# Mark and annotate intersection points
axes[1].scatter([k_g_cross], [GPM_THRESHOLD * 100], color="#1f77b4", s=60, zorder=5)
axes[1].scatter([k_h_cross], [GPM_THRESHOLD * 100], color="#d62728", s=60, zorder=5)

axes[1].annotate(
    f"K = {k_g_cross}",
    (k_g_cross, GPM_THRESHOLD * 100),
    textcoords="offset points",
    xytext=(10, -15),
    color="#1f77b4",
    fontweight="bold",
    fontsize=10,
)
axes[1].annotate(
    f"K = {k_h_cross}",
    (k_h_cross, GPM_THRESHOLD * 100),
    textcoords="offset points",
    xytext=(-45, 10),
    color="#d62728",
    fontweight="bold",
    fontsize=10,
)

axes[1].set_title(
    f"Cumulative Operator Variance Explained (Layer {FOCAL_LAYER + 1})",
    fontsize=12,
    fontweight="bold",
)
axes[1].set_xlabel("Mode Index $k$", fontsize=11)
axes[1].set_ylabel("Cumulative Variance Ratio (%)", fontsize=11)
axes[1].set_xlim(1, len(k_modes_focal))
axes[1].set_ylim(0, 105)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(loc="lower right", fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# --- 1. CONFIGURATION ---
TARGET_TASK = 0  # Task index to evaluate (e.g. Task 0)
BATCH_SIZE = 1024  # Batch of empirical MNIST digits
THRESHOLD_ENERGY = 0.97  # GPM basis energy retention (97%)

# Extract empirical MNIST batch for Task 0
# Shape: [Batch_Size, 784]
X_mnist_batch = test_imgs_raw[:BATCH_SIZE].to(DEVICE, dtype=torch.float32)


# --- 2. SIGNAL PROPAGATION & SPECTRAL ANALYSIS FUNCTION ---
def analyze_propagated_signals(
    jacobians_run: list,
    X_input: torch.Tensor,
    task_idx: int = 0,
    threshold: float = 0.97,
) -> dict:
    """Propagates empirical input batch X through cumulative Jacobians across depth

    and computes the SVD metrics of the resulting signal representations Z^l.
    """
    task_jacs = jacobians_run[task_idx]
    num_layers = len(task_jacs)

    stable_ranks = []
    basis_ranks_97 = []
    layer_spectra = []

    for l_idx in range(num_layers):
        J_cum = task_jacs[l_idx]["J_cum"].to(
            X_input.device, dtype=X_input.dtype
        )

        # 1. Linearized push-forward: Z^l = X^0 @ (J_{0->l})^T  -> Shape: [B, d_l]
        Z_l = X_input @ J_cum.T

        # 2. SVD of propagated signal matrix Z^l
        # svdvals returns singular values in descending order
        S = torch.linalg.svdvals(Z_l)
        S_np = S.cpu().numpy().astype(float)
        layer_spectra.append(S_np)

        # 3. Compute Stable Rank: sum(s_k^2) / s_1^2
        srank = np.sum(S_np**2) / (S_np[0] ** 2 + 1e-12)
        stable_ranks.append(srank)

        # 4. Compute Required Basis Rank K at 97% Energy
        cum_energy = np.cumsum(S_np**2) / np.sum(S_np**2)
        k_cross = (
            np.where(cum_energy >= threshold)[0][0] + 1
            if np.any(cum_energy >= threshold)
            else len(S_np)
        )
        basis_ranks_97.append(k_cross)

    return {
        "stable_ranks": stable_ranks,
        "basis_ranks_97": basis_ranks_97,
        "spectra": layer_spectra,
        "num_layers": num_layers,
    }


# Run propagation on both Gaussian and Heavy-Tailed runs
prop_gaussian = analyze_propagated_signals(
    jacobians_gaussian,
    X_mnist_batch,
    task_idx=TARGET_TASK,
    threshold=THRESHOLD_ENERGY,
)
prop_ht = analyze_propagated_signals(
    jacobians_ht,
    X_mnist_batch,
    task_idx=TARGET_TASK,
    threshold=THRESHOLD_ENERGY,
)

# --- 3. PLOT 3-PANEL COMPARATIVE FIGURE ---
layers_axis = np.arange(1, prop_gaussian["num_layers"] + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# PANEL 1: Stable Rank vs. Depth
axes[0].plot(
    layers_axis,
    prop_gaussian["stable_ranks"],
    label=r"Gaussian ($\alpha = 2.0$)",
    color="#1f77b4",
    linestyle="--",
    marker="o",
    lw=2.0,
)
axes[0].plot(
    layers_axis,
    prop_ht["stable_ranks"],
    label=r"Heavy-Tailed ($\alpha = 1.2$)",
    color="#d62728",
    linestyle="-",
    marker="s",
    lw=2.0,
)
axes[0].set_title(
    "Empirical Signal Stable Rank vs. Depth", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Layer Depth $l$", fontsize=11)
axes[0].set_ylabel(r"Stable Rank $\sum \sigma_k^2 / \sigma_1^2$", fontsize=11)
axes[0].set_xticks(layers_axis)
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(fontsize=10, loc="upper right")

# PANEL 2: Propagated Basis Rank (K at 97% Energy)
axes[1].plot(
    layers_axis,
    prop_gaussian["basis_ranks_97"],
    label=r"Gaussian ($\alpha = 2.0$)",
    color="#1f77b4",
    linestyle="--",
    marker="o",
    lw=2.0,
)
axes[1].plot(
    layers_axis,
    prop_ht["basis_ranks_97"],
    label=r"Heavy-Tailed ($\alpha = 1.2$)",
    color="#d62728",
    linestyle="-",
    marker="s",
    lw=2.0,
)
axes[1].set_title(
    r"Propagated Basis Rank ($K$ at 97% Energy)", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Layer Depth $l$", fontsize=11)
axes[1].set_ylabel("Required Basis Rank $K$", fontsize=11)
axes[1].set_xticks(layers_axis)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(fontsize=10, loc="upper right")

# PANEL 3: Transmitted Spectrum at Deepest Layer
# final_l = prop_gaussian["num_layers"] - 2
# s_g_final = prop_gaussian["spectra"][final_l]
# s_h_final = prop_ht["spectra"][final_l]

# s_g_norm = s_g_final / s_g_final[0]
# s_h_norm = s_h_final / s_h_final[0]
# k_modes = np.arange(1, len(s_g_norm) + 1)

# axes[2].plot(
#     k_modes,
#     s_g_norm,
#     label=f"Gaussian (Layer {final_l + 1})",
#     color="#1f77b4",
#     linestyle="--",
#     lw=2.0,
# )
# axes[2].plot(
#     k_modes,
#     s_h_norm,
#     label=f"Heavy-Tailed (Layer {final_l + 1})",
#     color="#d62728",
#     linestyle="-",
#     lw=2.0,
# )
# axes[2].set_xscale("log")
# axes[2].set_yscale("log")
# axes[2].set_title(
#     f"Transmitted MNIST Spectrum (Layer {final_l + 1})",
#     fontsize=12,
#     fontweight="bold",
# )
# axes[2].set_xlabel("Mode Index $k$", fontsize=11)
# axes[2].set_ylabel(r"Normalized Singular Value $s_k / s_1$", fontsize=11)
# axes[2].grid(True, which="both", linestyle="--", alpha=0.4)
# axes[2].legend(fontsize=10, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------------------------------------
# 1. EXTRACT SPECTRA & COMPUTE ENERGY FRACTIONS
# -------------------------------------------------------------
EVAL_LAYER = 8  # Penultimate feature layer (0-indexed Layer 8)
TOP_K_SCREE = 15  # Modes to display on the scree panel
MAX_K_CDF = 50  # Modes to display on the cumulative curve

# Extract singular values from pre-computed propagated runs
s_g = prop_gaussian["spectra"][EVAL_LAYER]
s_ht = prop_ht["spectra"][EVAL_LAYER]

# Fraction of total variance per singular mode
var_frac_g = (s_g**2) / np.sum(s_g**2) * 100.0
var_frac_ht = (s_ht**2) / np.sum(s_ht**2) * 100.0

# Cumulative energy (CDF)
cum_energy_g = np.cumsum(var_frac_g)
cum_energy_ht = np.cumsum(var_frac_ht)

# -------------------------------------------------------------
# 2. PLOTTING THE 2-PANEL FIGURE
# -------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2), dpi=150)

# PANEL 1: Standard Linear Scree Plot (Top K Modes)
modes_scree = np.arange(1, TOP_K_SCREE + 1)
bar_width = 0.38

ax1.bar(
    modes_scree - bar_width / 2,
    var_frac_g[:TOP_K_SCREE],
    width=bar_width,
    label=r"Gaussian ($\alpha=2.0$)",
    color="#1f77b4",
    alpha=0.85,
)
ax1.bar(
    modes_scree + bar_width / 2,
    var_frac_ht[:TOP_K_SCREE],
    width=bar_width,
    label=r"Heavy-Tailed ($\alpha=1.2$)",
    color="#d62728",
    alpha=0.85,
)

ax1.set_title(
    f"Singular Value Scree Plot (Layer {EVAL_LAYER + 1})",
    fontsize=12,
    fontweight="bold",
)
ax1.set_xlabel("Singular Mode Index $k$", fontsize=11)
ax1.set_ylabel("Variance Explained (%)", fontsize=11)
ax1.set_xticks(modes_scree)
ax1.grid(True, linestyle="--", alpha=0.4, axis="y")
ax1.legend(fontsize=10, loc="upper right")

# PANEL 2: Cumulative Variance Retention (CDF)
modes_cdf = np.arange(1, MAX_K_CDF + 1)

ax2.plot(
    modes_cdf,
    cum_energy_g[:MAX_K_CDF],
    color="#1f77b4",
    marker="o",
    markersize=3.5,
    lw=2.0,
    label=r"Gaussian ($\alpha=2.0$)",
)
ax2.plot(
    modes_cdf,
    cum_energy_ht[:MAX_K_CDF],
    color="#d62728",
    marker="s",
    markersize=3.5,
    lw=2.0,
    label=r"Heavy-Tailed ($\alpha=1.2$)",
)

# 97% Energy Threshold line used by GPM
ax2.axhline(
    97.0,
    color="black",
    linestyle=":",
    lw=1.5,
    label="97% GPM Basis Threshold",
)

# Annotate crossing points
k_g_97 = np.where(cum_energy_g >= 97.0)[0][0] + 1
k_ht_97 = np.where(cum_energy_ht >= 97.0)[0][0] + 1

ax2.scatter(
    [k_g_97],
    [cum_energy_g[k_g_97 - 1]],
    color="#1f77b4",
    s=60,
    zorder=5,
)
ax2.scatter(
    [k_ht_97],
    [cum_energy_ht[k_ht_97 - 1]],
    color="#d62728",
    s=60,
    zorder=5,
)

# ax2.annotate(
#     f"Gaussian: $K={k_g_97}$",
#     xy=(k_g_97, 97.0),
#     xytext=(k_g_97 - 8, 80.0),
#     arrowprops=dict(arrowstyle="->", color="#1f77b4", lw=1.2),
#     fontsize=9,
#     fontweight="semibold",
#     color="#1f77b4",
# )
# ax2.annotate(
#     f"HT: $K={k_ht_97}$",
#     xy=(k_ht_97, 97.0),
#     xytext=(k_ht_97 + 3, 88.0),
#     arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.2),
#     fontsize=9,
#     fontweight="semibold",
#     color="#d62728",
# )

ax2.set_title(
    f"Cumulative Subspace Energy Retention (Layer {EVAL_LAYER + 1})",
    fontsize=12,
    fontweight="bold",
)
ax2.set_xlabel("Number of Stored Modes ($K$)", fontsize=11)
ax2.set_ylabel("Cumulative Energy Explained (%)", fontsize=11)
ax2.set_ylim([0, 105])
ax2.set_xlim([1, MAX_K_CDF])
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(fontsize=10, loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------------------------------------
# 1. COMPUTE TRANSVERSE DISPERSION (R_perp) & ANCHOR TRAJECTORIES
# -------------------------------------------------------------
def extract_conduit_metrics(activations_list):
    pr_list = []
    eta_perp_list = []
    anchor_norms = []

    for l_idx, h_all in enumerate(activations_list):
        h_center = h_all[0]
        h_cloud = h_all[1:]
        delta_h = h_cloud - h_center

        _, S, _ = np.linalg.svd(delta_h, full_matrices=False)
        eigenvals = (S**2) / (len(h_cloud) - 1)

        # 1. Scale-invariant Participation Ratio
        sum_e = np.sum(eigenvals)
        sum_sq_e = np.sum(eigenvals**2)
        pr = (sum_e**2) / (sum_sq_e + 1e-12)
        pr_list.append(pr)

        # 2. Scale-invariant Relative Transverse Fraction
        # Measures fraction of total energy leaking out of dominant mode
        frac_transverse = np.sqrt(
            np.sum(eigenvals[1:]) / (sum_e + 1e-12)
        )
        eta_perp_list.append(frac_transverse)

        anchor_norms.append(np.linalg.norm(h_center))

    return {
        "PR": np.array(pr_list),
        "R_perp": np.array(eta_perp_list),  # Normalized to [0, 1]
        "anchor": np.array(anchor_norms),
    }


c_g = extract_conduit_metrics(acts_gauss)
c_ht = extract_conduit_metrics(acts_ht)

# Normalize anchor trajectories to start at 0 so Y-axis represents relative depth transport
anchor_base_g = c_g["anchor"] - c_g["anchor"][0]
anchor_base_ht = c_ht["anchor"] - c_ht["anchor"][0]

# -------------------------------------------------------------
# 2. PLOTTING THE 3-PANEL HERO FIGURE
# -------------------------------------------------------------
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.5), dpi=160)
fig.suptitle(
    "Perturbation Conduit Dynamics: Transverse Null-Space Dispersion vs."
    " Channelization",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

# Common limits for Panels 1 and 2
y_all_min = min((anchor_base_g - c_g["R_perp"]).min(), (anchor_base_ht - c_ht["R_perp"]).min()) * 1.2
y_all_max = max((anchor_base_g + c_g["R_perp"]).max(), (anchor_base_ht + c_ht["R_perp"]).max()) * 1.2

# --- PANEL 1: Gaussian Baseline (Porous, Space-Filling Conduit) ---
ax1.fill_between(
    depth_axis,
    anchor_base_g - c_g["R_perp"],
    anchor_base_g + c_g["R_perp"],
    color="#1f77b4",
    alpha=0.3,
    label=r"Transverse Spread $R_\perp = \sqrt{\sum_{k 2} s_k}$",
)
ax1.plot(
    depth_axis,
    anchor_base_g + c_g["R_perp"],
    color="#1f77b4",
    linestyle="--",
    lw=1.2,
    alpha=0.8,
)
ax1.plot(
    depth_axis,
    anchor_base_g - c_g["R_perp"],
    color="#1f77b4",
    linestyle="--",
    lw=1.2,
    alpha=0.8,
)
ax1.plot(
    depth_axis,
    anchor_base_g,
    color="#08306b",
    lw=2.5,
    marker="o",
    markersize=4,
    label="Anchor Trajectory",
    zorder=5,
)

ax1.set_title(
    "Gaussian Baseline ($\\alpha = 2.0$)\nDiffuse Conduit (Leaking into Ambient Subspaces)",
    fontsize=11,
    fontweight="semibold",
)
ax1.set_xlabel("Network Layer Depth $l$", fontsize=10)
ax1.set_ylabel("Latent Feature Displacement", fontsize=10)
ax1.set_xticks(depth_axis)
ax1.set_xticklabels(depth_labels)
ax1.set_ylim([y_all_min, y_all_max])
ax1.grid(True, linestyle="--", alpha=0.35)
ax1.legend(loc="upper left", fontsize=8.5, framealpha=0.9)

# --- PANEL 2: Heavy-Tailed (Immediate Layer 1 Pinch) ---
ax2.fill_between(
    depth_axis,
    anchor_base_ht - c_ht["R_perp"],
    anchor_base_ht + c_ht["R_perp"],
    color="#d62728",
    alpha=0.3,
    label=r"Transverse Spread $R_\perp = \sqrt{\sum_{k 2} s_k}$",
)
ax2.plot(
    depth_axis,
    anchor_base_ht + c_ht["R_perp"],
    color="#d62728",
    linestyle="--",
    lw=1.2,
    alpha=0.8,
)
ax2.plot(
    depth_axis,
    anchor_base_ht - c_ht["R_perp"],
    color="#d62728",
    linestyle="--",
    lw=1.2,
    alpha=0.8,
)
ax2.plot(
    depth_axis,
    anchor_base_ht,
    color="#67000d",
    lw=2.5,
    marker="s",
    markersize=4,
    label="Anchor Trajectory",
    zorder=5,
)

ax2.set_title(
    "Heavy-Tailed ($\\alpha = 1.2$)\nChannelized Ribbon (Transverse Bulk Quenched to Zero)",
    fontsize=11,
    fontweight="semibold",
)
ax2.set_xlabel("Network Layer Depth $l$", fontsize=10)
ax2.set_xticks(depth_axis)
ax2.set_xticklabels(depth_labels)
ax2.set_ylim([y_all_min, y_all_max])
ax2.grid(True, linestyle="--", alpha=0.35)
ax2.legend(loc="upper left", fontsize=8.5, framealpha=0.9)

# --- PANEL 3: Participation Ratio vs. Depth (Retained) ---
ax3.plot(
    depth_axis,
    c_g["PR"],
    label=r"Gaussian ($\alpha=2.0$)",
    color="#1f77b4",
    marker="o",
    lw=2.0,
)
ax3.plot(
    depth_axis,
    c_ht["PR"],
    label=r"Heavy-Tailed ($\alpha=1.2$)",
    color="#d62728",
    marker="s",
    lw=2.0,
)

ax3.set_title(
    "Perturbation Cloud Effective Dimension Across Depth",
    fontsize=11,
    fontweight="semibold",
)
ax3.set_xlabel("Network Layer Depth $l$", fontsize=10)
ax3.set_ylabel(r"Participation Ratio $\operatorname{PR}(l)$", fontsize=10)
ax3.set_xticks(depth_axis)
ax3.set_xticklabels(depth_labels)
ax3.grid(True, linestyle="--", alpha=0.4)
ax3.legend(fontsize=9, loc="upper right")

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np

# Select layers to track across the 9-layer backbone
TRACK_LAYERS = [0, 2, 4, 6, 8]  # Layers 1, 3, 5, 7, 9 (0-indexed)
NUM_MODES_TO_SHOW = 40

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=160, sharey=True)
fig.suptitle(
    "Depth-Wise Spectral Evolution: Parallel Translation vs. Anisotropic"
    " Steepening",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

# Colormap: cool (shallow layers) to warm (deep layers)
colors = cm.plasma(np.linspace(0.15, 0.9, len(TRACK_LAYERS)))
k_axis = np.arange(1, NUM_MODES_TO_SHOW + 1)

for idx, l_idx in enumerate(TRACK_LAYERS):
    layer_num = l_idx + 1
    c = colors[idx]

    # Extract raw, unnormalized singular values
    s_g = prop_gaussian["spectra"][l_idx][:NUM_MODES_TO_SHOW]
    s_ht = prop_ht["spectra"][l_idx][:NUM_MODES_TO_SHOW]

    ax1.plot(
        k_axis,
        s_g,
        color=c,
        lw=1.8,
        marker="o",
        markersize=3,
        label=f"Layer {layer_num}",
    )
    ax2.plot(
        k_axis,
        s_ht,
        color=c,
        lw=2.0,
        marker="s",
        markersize=3,
        label=f"Layer {layer_num}",
    )

# Gaussian Panel
ax1.set_title(
    "Gaussian Baseline ($\\alpha = 2.0$)\nHomogeneous Decay (Preserves Flat"
    " Slope)",
    fontsize=11,
    fontweight="semibold",
)
ax1.set_xlabel("Singular Mode Index $k$", fontsize=11)
ax1.set_ylabel(
    r"Raw Singular Value $s_k(Z^l)$ (Log Scale)",
    fontsize=11,
)
ax1.set_yscale("log")
ax1.grid(True, which="both", linestyle="--", alpha=0.35)
ax1.legend(loc="lower left", fontsize=9, framealpha=0.9)

# Heavy-Tailed Panel
ax2.set_title(
    "Heavy-Tailed ($\\alpha = 1.2$)\nRotational Steepening (Power-Law"
    " Condensation)",
    fontsize=11,
    fontweight="semibold",
)
ax2.set_xlabel("Singular Mode Index $k$", fontsize=11)
ax2.set_yscale("log")
ax2.grid(True, which="both", linestyle="--", alpha=0.35)
ax2.legend(loc="lower left", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()